In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
from sklearn.metrics import confusion_matrix
from IPython.display import display
from sklearn.metrics import precision_score, recall_score, f1_score
import statsmodels.api as sm
from statsmodels.formula.api import ols


In [ ]:
MONGODB_HOST = 'your host'
MONGODB_PORT = 27017
MONGODB_NAME = 'RAGQA'

# Función para obtener la base de datos
def get_db():
    client = MongoClient(MONGODB_HOST, MONGODB_PORT)
    return client[MONGODB_NAME]

def load_data(collection):
    db = get_db()
    eval_collection = db[collection]
    eval_data = list(eval_collection.find({}))
    return pd.DataFrame(eval_data)


In [ ]:
EVAL_COLLECTION_LIST_SUPERVISED = ["ia_eval_supervised", "ia_deepseek_eval_supervised", "ia_deepseek-r1_70b_eval_supervised", "ia_phi4_eval_supervised", "ia_qwen_2_5_eval_supervised", "eval_supervised"]
EVAL_COLLECTION_LIST = [ "ia_eval", "ia_deepseek_eval", "ia_deepseek-r1_70b_eval", "ia_phi_4_eval", "ia_qwen_2_5_eval", "eval"]
EVAL_NAMES = [ "Llama3.1:7b", "Deepseek-r1:7b", "Deepseek-r1:70b", "Phi4", "Qwen2.5", "Human"]
EVAL_COLLECTION_LIST_BIASED = ["ia_biased_eval", "ia_biased_deepseek_eval", "ia_biased_deepseek-r1_70b_eval", "ia_biased_phi4_eval", "ia_biased_qwen_2_5_eval", "biased_eval"]


In [ ]:
def create_connection_data(df_score, x_positions, llm_types):
    """
    Crea los datos para las líneas de conexión en el orden correcto.
    - Conecta Human (Start) -> Basic AI -> Advanced AI.
    - Conecta Advanced AI -> Human (End) si Human (End) está presente.
    """
    connections = []
    for idx in df_score.index:
        # Obtener los valores originales (sin duplicar)
        original_values = df_score.loc[idx, ['Human (Start)', 'Basic AI', 'Advanced AI', 'Human (End)']].dropna()
        
        if len(original_values) >= 2:
            # Conexión entre Human (Start), Basic AI y Advanced AI
            if 'Human (Start)' in original_values and 'Basic AI' in original_values:
                x_values = [x_positions['Human (Start)'], x_positions['Basic AI']]
                y_values = [original_values['Human (Start)'], original_values['Basic AI']]
                connections.append((x_values, y_values))
            
            if 'Basic AI' in original_values and 'Advanced AI' in original_values:
                x_values = [x_positions['Basic AI'], x_positions['Advanced AI']]
                y_values = [original_values['Basic AI'], original_values['Advanced AI']]
                connections.append((x_values, y_values))
            
            # Conexión entre Advanced AI y Human (End)
            if 'Advanced AI' in original_values and 'Human (End)' in original_values:
                x_values = [x_positions['Advanced AI'], x_positions['Human (End)']]
                y_values = [original_values['Advanced AI'], original_values['Human (End)']]
                connections.append((x_values, y_values))
    
    return connections

In [ ]:
def filter_and_analyze_responses(df):
    patterns = [
        r"i\s*(?:do\s*not|don'?t)\s*know",
        r"no\s*(?:lo)?\s*s[ée]",
 
    ]
    
    mask = df['original_answer'].str.lower().str.contains('|'.join(patterns), regex=True, na=False)
    df_filtered = df[~mask].copy()
    removed_counts = (
        df[mask]
        .groupby(['LLM', 'original_answer_id'])
        .size()
        .reset_index(name='removed_count')
    )
    
    class_summary = (
        removed_counts
        .groupby('LLM')
        .agg({
            'original_answer_id': 'nunique',
            'removed_count': 'sum'
        })
        .rename(columns={
            'original_answer_id': 'unique_answers_removed',
            'removed_count': 'total_responses_removed'
        })
    )
    
    class_mapping = {
        0: 'Humano',
        1: 'IA',
        2: 'IA Avanzada'
    }
    
    class_summary.index = class_summary.index.map(class_mapping)
    
    return df_filtered, class_summary, removed_counts

In [ ]:
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from textstat import flesch_reading_ease, syllable_count, lexicon_count, text_standard, automated_readability_index
import re

# Cargar los modelos de spaCy (asegúrate de tener instalado 'es_core_news_md' o 'en_core_web_md')
# !python -m spacy download es_core_news_md  # Descomentar si necesitas español
!python -m spacy download en_core_web_md   # Descomentar si necesitas inglés
nlp = spacy.load('en_core_web_md')  # Cambia a 'en_core_web_md' para inglés

# Asumiendo que df_human y df_ai son tus dataframes con las evaluaciones
# df_human = pd.read_csv('evaluaciones_humanas.csv')
# df_ai = pd.read_csv('evaluaciones_ia.csv')

def extract_linguistic_features(text):
    """Extrae características lingüísticas de un texto."""
    if pd.isna(text) or text == "":
        return {
            'word_count': 0,
            'sentence_count': 0,
            'avg_word_length': 0,
            'avg_sentence_length': 0,
            'unique_words_ratio': 0,
            'lexical_diversity': 0,
            'reading_ease': 0,
            "grade_level": 0,
            'verb_ratio': 0,
            'noun_ratio': 0,
            'adj_ratio': 0,
            'adv_ratio': 0,
            'pronoun_ratio': 0,
            'named_entity_count': 0,
            'dependency_distance': 0,
            'connector_count': 0
        }
    
    doc = nlp(text)
   
    # Conteos básicos
    words = [token.text for token in doc if not token.is_punct and not token.is_space]
    
    word_count = len(words)
    sentence_count = len(list(doc.sents))
    
    if word_count == 0 or sentence_count == 0:
        return {
            'word_count': 0,
            'sentence_count': 0,
            'avg_word_length': 0,
            'avg_sentence_length': 0,
            'unique_words_ratio': 0,
            'lexical_diversity': 0,
            'reading_ease': 0,
            "grade_level": 0,
            'readind_ease': 0, 
            'verb_ratio': 0,
            'noun_ratio': 0,
            'adj_ratio': 0,
            'adv_ratio': 0,
            'pronoun_ratio': 0,
            'named_entity_count': 0,
            'dependency_distance': 0,
            'connector_count': 0
        }
    
    # Promedios y ratios
    avg_word_length = sum(len(word) for word in words) / word_count
    avg_sentence_length = word_count / sentence_count
    unique_words = set(word.lower() for word in words)
    unique_words_ratio = len(unique_words) / word_count
    
    # Tipo de palabras
    pos_counts = Counter([token.pos_ for token in doc])
    verb_count = pos_counts.get('VERB', 0)
    noun_count = pos_counts.get('NOUN', 0)
    adj_count = pos_counts.get('ADJ', 0)
    adv_count = pos_counts.get('ADV', 0)
    pronoun_count = pos_counts.get('PRON', 0)
    
    # Entidades nombradas
    named_entity_count = len(doc.ents)
    
    # Distancia de dependencia (promedio de la distancia entre tokens relacionados)
    dependency_distances = [abs(token.i - token.head.i) for token in doc if token.head is not token]
    avg_dependency_distance = sum(dependency_distances) / len(dependency_distances) if dependency_distances else 0
    
    # Conteo de conectores (ajustar según el idioma)
    connectors = ['however', 'therefore', 'furthermore', 'moreover', 'on the other hand', 
                  'consequently', 'although', 'because', 'since', 'thus', 'nevertheless', 
                  'in addition', 'as a result', 'despite', 'in contrast', 'for example',
                  'in fact', 'specifically', 'in conclusion', 'in summary']
    connector_count = sum(1 for conn in connectors if conn in text.lower())
    
    # Métricas de legibilidad
    reading_ease = flesch_reading_ease(text)  # Nota: esto funciona mejor en inglés
    grade = automated_readability_index(text)
    return {
        'word_count': word_count,
        'sentence_count': sentence_count,
        'avg_word_length': avg_word_length,
        'avg_sentence_length': avg_sentence_length,
        'unique_words_ratio': unique_words_ratio,
        'lexical_diversity': len(unique_words) / (word_count ** 0.5),  # MTLD normalizado
        'reading_ease': reading_ease,
        "grade_level": grade,
        'verb_ratio': verb_count / word_count,
        'noun_ratio': noun_count / word_count,
        'adj_ratio': adj_count / word_count,
        'adv_ratio': adv_count / word_count,
        'pronoun_ratio': pronoun_count / word_count,
        'named_entity_count': named_entity_count,
        'dependency_distance': avg_dependency_distance,
        'connector_count': connector_count
    }

def analyze_dataframe(df):
    """Analiza características lingüísticas para un dataframe."""
    # Añadir columnas para cada característica lingüística
    linguistic_features = df['response'].apply(extract_linguistic_features)
    
    # Convertir la lista de diccionarios en columnas
    for feature in linguistic_features.iloc[0].keys():
        df[feature] = linguistic_features.apply(lambda x: x[feature])
    
    # Agrupar por fuente (LLM) y calcular medias
    feature_means = df.groupby('LLM')[list(linguistic_features.iloc[0].keys())].mean()
    
    # Analizar correlación entre características y puntuación
    correlations = {}
    for feature in linguistic_features.iloc[0].keys():
        correlations[feature] = df[[feature, 'accuracy_score']].corr().iloc[0, 1]
    
    return feature_means, correlations

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def plot_feature_comparison(feature_means):
    """Generate comparative plots for linguistic features using Plotly with each feature in a separate subplot."""
    
    # Create response type labels mapping
    label_mapping = {0: 'Human', 1: 'AI', 2: 'AI CoT'}
    
    # Color palette
    colors = {
        'Human': 'rgb(31, 119, 180)',
        'AI': 'rgb(255, 127, 14)',
        'AI CoT': 'rgb(44, 160, 44)'
    }
    
    # Map the index to readable labels
    feature_means_labeled = feature_means.copy()
    feature_means_labeled.index = [label_mapping.get(idx, f'Type {idx}') for idx in feature_means.index]
    
    # Get features and clean their names for display
    features = list(feature_means.columns)
    feature_labels = [feature.replace('_', ' ').title() for feature in features]
    
    # Calculate subplot layout
    n_features = len(features)
    cols = 4
    rows = (n_features + cols - 1) // cols
    
    # Create subplots
    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=feature_labels,
        vertical_spacing=0.08,
        horizontal_spacing=0.08
    )
    
    # Add bars for each feature
    for i, feature in enumerate(features):
        row = i // cols + 1
        col = i % cols + 1
        
        # Add bars for each response type
        for j, response_type in enumerate(feature_means_labeled.index):
            fig.add_trace(
                go.Bar(
                    x=[response_type],
                    y=[feature_means_labeled.loc[response_type, feature]],
                    name=response_type,
                    marker_color=colors[response_type],
                    text=[f'{feature_means_labeled.loc[response_type, feature]:.3f}'],
                    textposition='outside',
                    textfont=dict(size=18, color='black'),
                    opacity=0.8,
                    showlegend=(i == 0),  # Only show legend for first subplot
                    legendgroup=response_type
                ),
                row=row, col=col
            )
        
        # Calculate y-axis range with extra space for text labels
        max_val = feature_means_labeled[feature].max()
        min_val = feature_means_labeled[feature].min()
        if min_val >= 0:
            y_range = [0, max_val * 1.15]  # Add 15% extra space at the top
        else:
            y_range = [min_val * 1.3, max_val * 1.15]  # Add 15% extra space at the top
        # Update axes for each subplot
        fig.update_xaxes(
            tickfont=dict(size=18, color='black', family='Arial'),
            showgrid=False,
            gridcolor='lightgray',
            gridwidth=0,
            showline=True,
            linewidth=2,
            linecolor='black',
            showticklabels=False,
            row=row, col=col
        )
        fig.update_yaxes(
            range=y_range,
            tickfont=dict(size=18, color='black', family='Arial'),
            showgrid=False,
            gridcolor='lightgray',
            gridwidth=0,
            showline=True,
            linewidth=2,
            linecolor='black',
            showticklabels=False,
            row=row, col=col
        )
    
    # Update layout for paper-like appearance
    fig.update_layout(
        title=dict(
            text='<b>Linguistic Feature Comparison Across Response Types</b>',
            x=0.5,
            y=1,
            font=dict(size=30, color='black', family='Arial')
        ),
        legend=dict(
            font=dict(size=22, color='black', family='Arial'),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='black',
            borderwidth=1,
            x=1.02,
            y=1.02,
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(family='Arial', size=20, color='black'),
        width=1000,
        height=250 * rows,
        margin=dict(l=80, r=150, t=100, b=80)
    )
    
    # Update subplot titles font and position
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=20, color='black', family='Arial')
        annotation['y'] = annotation['y'] + 0.01  # Move titles up
    
    fig.show()
    return fig

def analyze_text_complexity(df):
    """Analiza y compara la complejidad del texto entre diferentes fuentes."""
    # Combinar ambos dataframes
    df_combined = df
    
    # Extraer características para cada respuesta
    feature_means, correlations = analyze_dataframe(df_combined)
    
    # Visualizar diferencias
    fig = plot_feature_comparison(feature_means)
    
    # Analizar correlaciones entre características y puntuaciones
    sorted_correlations = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    
    # Análisis de TF-IDF para términos distintivos
    tfidf = TfidfVectorizer(max_features=100, stop_words='english') 
    
    results = {
        'feature_means': feature_means,
        'correlations': sorted_correlations,
        'visualization': fig
    }
    
    # Para cada tipo de fuente, identificar términos característicos
    source_terms = {}
    for source in df_combined['LLM'].unique():
        source_texts = df_combined[df_combined['LLM'] == source]['response'].dropna()
        if len(source_texts) > 0:
            try:
                source_matrix = tfidf.fit_transform(source_texts)
                feature_names = tfidf.get_feature_names_out()
                
                # Obtener los términos más importantes para esta fuente
                if source_matrix.shape[0] > 0:
                    importance = np.asarray(source_matrix.mean(axis=0)).flatten()
                    top_indices = importance.argsort()[-10:][::-1]
                    top_terms = [(feature_names[i], importance[i]) for i in top_indices]
                    source_terms[source] = top_terms
            except:
                source_terms[source] = []
    
    results['distinctive_terms'] = source_terms
    
    return results

# Función principal para ejecutar el análisis
def run_linguistic_analysis(df):
    print("Iniciando análisis lingüístico comparativo...")
    results = analyze_text_complexity(df)
    
    print("\nCaracterísticas lingüísticas promedio por fuente:")
    print(results['feature_means'])
    
    print("\nCorrelaciones entre características lingüísticas y puntuaciones:")
    for feature, corr in results['correlations']:
        print(f"{feature}: {corr:.4f}")
    
    print("\nTérminos distintivos por fuente:")
    for source, terms in results['distinctive_terms'].items():
        print(f"\nFuente {source}:")
        for term, importance in terms:
            print(f"  - {term}: {importance:.4f}")
    
    return results

In [ ]:
def plot_feature_comparison(feature_means):
    """Generate radar chart for linguistic features comparison across response types."""
    
    # Create response type labels mapping
    label_mapping = {0: 'Human', 1: 'AI', 2: 'AI CoT'}
    
    # Color palette
    colors = {
        'Human': 'rgb(31, 119, 180)',
        'AI': 'rgb(255, 127, 14)',
        'AI CoT': 'rgb(44, 160, 44)'
    }
    
    # Map the index to readable labels
    feature_means_labeled = feature_means.copy()
    feature_means_labeled.index = [label_mapping.get(idx, f'Type {idx}') for idx in feature_means.index]
    
    # Get features and clean their names for display
    features = list(feature_means.columns)
    
    # Group features by category for better visual organization
    feature_groups = {
        'Basic Metrics': ['word_count', 'sentence_count', 'avg_word_length', 'avg_sentence_length'],
        'Lexical Features': ['unique_words_ratio', 'lexical_diversity'],
        'Readability': ['reading_ease', 'grade_level'],
        'Part-of-Speech': ['verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio'],
        'Complexity': ['named_entity_count', 'dependency_distance', 'connector_count']
    }
    
    # Reorder features by groups for better visual flow
    ordered_features = []
    for group in feature_groups.values():
        for feature in group:
            if feature in features:
                ordered_features.append(feature)
    
    # Add any remaining features not in groups
    for feature in features:
        if feature not in ordered_features:
            ordered_features.append(feature)
    
    # Create ordered feature labels
    feature_labels = [feature.replace('_', ' ').title() for feature in ordered_features]
    
    # Normalize the data to 0-1 scale for better radar visualization
    # Apply MinMaxScaler separately for each feature with min=0
    feature_means_normalized = feature_means_labeled.copy()
    for feature in ordered_features:
        max_val = feature_means_labeled[feature].max()
        min_val = 0  # Always use 0 as minimum
        feature_means_normalized[feature] = (feature_means_labeled[feature] - min_val) / (max_val - min_val)
    
    # Create the radar chart
    fig = go.Figure()
    
    # Add trace for each response type
    for response_type in feature_means_normalized.index:
        values = [feature_means_normalized.loc[response_type, feature] for feature in ordered_features]
        original_values = [feature_means_labeled.loc[response_type, feature] for feature in ordered_features]
        # Close the radar chart by adding the first value at the end
        values += [values[0]]
        original_values += [original_values[0]]
        theta_labels = feature_labels + [feature_labels[0]]
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=theta_labels,
            fill='toself',
            name=response_type,
            line=dict(color=colors[response_type], width=3),
            fillcolor=colors[response_type].replace('rgb', 'rgba').replace(')', ', 0.1)'),
            marker=dict(
                size=8,
                color=colors[response_type],
                line=dict(color='white', width=2)
            ),
            hovertemplate=(
                f'<b>{response_type}</b><br>' +
                'Feature: %{theta}<br>' +
                'Normalized Value: %{r:.3f}<br>' +
                'Original Value: %{customdata:.3f}<extra></extra>'
            ),
            customdata=original_values
        ))
    
    # Update layout for paper-like appearance
    fig.update_layout(
        title=dict(
            text='<b>Linguistic Features Profile Comparison</b>',
            x=0.5,
            font=dict(size=24, color='black', family='Arial')
        ),
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1],
                tickfont=dict(size=14, color='black', family='Arial'),
                gridcolor='lightgray',
                gridwidth=1,
                linecolor='black',
                linewidth=1
            ),
            angularaxis=dict(
                tickfont=dict(size=14, color='black', family='Arial'),
                linecolor='black',
                linewidth=1,
                gridcolor='lightgray',
                gridwidth=1
            ),
            bgcolor='white'
        ),
        legend=dict(
            font=dict(size=16, color='black', family='Arial'),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='black',
            borderwidth=1,
            x=1.05,
            y=1
        ),
        font=dict(family='Arial', size=14, color='black'),
        width=900,
        height=800,
        margin=dict(l=80, r=150, t=100, b=80),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    
    # Add annotation explaining normalization
    fig.add_annotation(
        text="<i>Note: Values normalized to 0-1 scale for comparison</i>",
        xref="paper", yref="paper",
        x=0.5, y=-0.05,
        showarrow=False,
        font=dict(size=12, color='gray', family='Arial'),
        xanchor='center'
    )
    fig.show()
    return fig

In [ ]:
# Cargar los datos
answers = load_data("answers")
evaluations = load_data("ia_eval")

# Convertir los IDs a string para asegurar compatibilidad
answers["_id"] = answers["_id"].astype(str)
evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)

# Método 1: Usando merge con todas las columnas necesarias
evaluations_with_answers = evaluations.merge(
    answers[['_id', 'answer', 'bibliography_sources', 'confidence_level']], 
    left_on='original_answer_id',
    right_on='_id',
    how='left'
)

# Función para combinar respuesta y bibliografía cuando esté disponible
def combine_answer_and_bibliography(row):
    if pd.notna(row['bibliography_sources']):
        return row['answer'] + '\n\nReferences: ' + row['bibliography_sources']
    else:
        return row['answer']

# Crear la columna original_answer con la combinación de respuesta y bibliografía
evaluations_with_answers['original_answer'] = evaluations_with_answers.apply(combine_answer_and_bibliography, axis=1)


# Verificar el resultado
print("Primeras filas del DataFrame final:")
print(evaluations_with_answers.head())
evaluations_with_answers["response"] = evaluations_with_answers["original_answer"]
evaluations_with_answers_ia, _ , _ = filter_and_analyze_responses(evaluations_with_answers)


In [ ]:

results = run_linguistic_analysis(evaluations_with_answers_ia)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler

def create_correlation_heatmap_scaled(df):
    """
    Crea un heatmap de correlaciones con características escaladas entre 0 y 1.
    Categorías: Conteo, Superficie y Estructura.
    
    Args:
        df: DataFrame con características lingüísticas y puntuaciones
        
    Returns:
        Dict con resultados y visualizaciones
    """
    # Identificar todas las columnas de puntuación
    score_columns = ["accuracy_score", "clarity_score", "completeness_score"]
    
    # Definir las categorías de características
    count_features = ['word_count', 'sentence_count']
    
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Todas las características
    all_features = count_features + surface_features + structure_features
    
    # Verificar qué características están disponibles
    available_features = [feature for feature in all_features if feature in df.columns]
    
    if not available_features:
        print("No se encontraron características lingüísticas en el dataframe.")
        return None
    
    if not score_columns:
        print("No se encontraron columnas de puntuación (_score) en el dataframe.")
        return None
    
    # Crear una copia del dataframe para no modificar el original
    df_scaled = df.copy()
    
    # Escalar todas las características lingüísticas entre 0 y 1
    print("Escalando características lingüísticas entre 0 y 1...")
    scaler = MinMaxScaler()
    if len(available_features) > 0:
        df_scaled[available_features] = scaler.fit_transform(df[available_features])
    
    # Calcular matriz de correlación con valores escalados
    correlation_matrix = df_scaled[available_features + score_columns].corr()
    
    # Ordenar características por categoría para visualización
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        ordered_features.extend([f for f in feature_group if f in available_features])
    
    # Extraer subconjunto de la matriz de correlación
    correlation_subset = correlation_matrix.loc[ordered_features, score_columns]
    
    # Crear figura para el heatmap
    plt.figure(figsize=(12, len(available_features) / 2))
    
    # Crear heatmap con anotaciones
    heatmap = sns.heatmap(
        correlation_subset, 
        annot=True,
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        center=0,
        fmt='.2f',
        linewidths=.5
    )
    
    # Añadir líneas para separar categorías
    count_end = sum(1 for feature in count_features if feature in ordered_features)
    surface_end = count_end + sum(1 for feature in surface_features if feature in ordered_features)
    
    plt.axhline(y=count_end, color='black', linewidth=2)
    plt.axhline(y=surface_end, color='black', linewidth=2)
    
    # Añadir etiquetas de categoría
    plt.text(-2.5, count_end / 2, 'CONTEO', verticalalignment='center', fontsize=12, fontweight='bold')
    plt.text(-2.5, count_end + (surface_end - count_end) / 2, 'SUPERFICIE', verticalalignment='center', fontsize=12, fontweight='bold')
    plt.text(-2.5, surface_end + (len(ordered_features) - surface_end) / 2, 'ESTRUCTURA', verticalalignment='center', fontsize=12, fontweight='bold')
    
    plt.title('Correlaciones entre Características Lingüísticas Escaladas y Puntuaciones', fontsize=16)
    plt.tight_layout()
    
    # Calcular correlaciones promedio por grupo
    avg_correlations = {
        'Conteo': pd.DataFrame({
            score: [abs(correlation_subset.loc[feature, score]) for feature in count_features if feature in available_features]
            for score in score_columns
        }).mean(),
        
        'Superficie': pd.DataFrame({
            score: [abs(correlation_subset.loc[feature, score]) for feature in surface_features if feature in available_features]
            for score in score_columns
        }).mean(),
        
        'Estructura': pd.DataFrame({
            score: [abs(correlation_subset.loc[feature, score]) for feature in structure_features if feature in available_features]
            for score in score_columns
        }).mean()
    }
    
    # Crear figura para el heatmap de correlaciones promedio
    fig_avg, ax_avg = plt.subplots(figsize=(10, 6))
    avg_df = pd.DataFrame(avg_correlations)
    
    sns.heatmap(avg_df.T, annot=True, cmap='coolwarm', fmt='.3f', linewidths=.5, ax=ax_avg, vmin=-1, vmax=1)
    ax_avg.set_title('Correlación Promedio por Categoría de Características Escaladas', fontsize=14)
    
    # Crear gráfico de barras para comparación visual
    fig_bar, ax_bar = plt.subplots(figsize=(12, 6))
    avg_df.plot(kind='bar', ax=ax_bar)
    ax_bar.set_title('Correlación Promedio por Tipo de Puntuación (Características Escaladas)', fontsize=14)
    ax_bar.set_ylabel('Correlación Promedio (valor absoluto)')
    ax_bar.set_xlabel('Tipo de Puntuación')
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # Crear un análisis explicativo de los resultados
    explanations = []
    for score in score_columns:
        category_correlations = {cat: val for cat, val in zip(avg_df.columns, avg_df.loc[score])}
        top_category = max(category_correlations, key=category_correlations.get)
        
        explanation = f"Para {score}, la categoría con mayor correlación es {top_category} ({category_correlations[top_category]:.3f}). "
        
        # Información sobre las características específicas con mayor correlación
        top_features = correlation_subset[score].abs().sort_values(ascending=False).head(3)
        explanation += f"Las características más correlacionadas son: "
        
        for i, (feature, corr_val) in enumerate(zip(top_features.index, correlation_subset.loc[top_features.index, score])):
            # Determinar a qué categoría pertenece esta característica
            if feature in count_features:
                cat = "Conteo"
            elif feature in surface_features:
                cat = "Superficie"
            else:
                cat = "Estructura"
                
            explanation += f"{feature} ({cat}, {corr_val:.3f})"
            if i < len(top_features) - 1:
                explanation += ", "
        
        explanations.append(explanation)
    
    # Análisis de dominancia entre categorías
    dominance_analysis = ""
    all_scores_avg = avg_df.mean()
    dominant_category = all_scores_avg.idxmax()
    
    dominance_analysis += f"\nEn promedio para todas las puntuaciones, la categoría {dominant_category} "
    dominance_analysis += f"tiene la mayor correlación ({all_scores_avg[dominant_category]:.3f}), "
    
    for cat in all_scores_avg.index:
        if cat != dominant_category:
            diff = all_scores_avg[dominant_category] - all_scores_avg[cat]
            dominance_analysis += f"superando a {cat} por {diff:.3f} ({all_scores_avg[cat]:.3f}). "
    
    return {
        'heatmap_fig': plt.gcf(),
        'avg_heatmap_fig': fig_avg,
        'bar_fig': fig_bar,
        'correlation_matrix': correlation_subset,
        'avg_correlations': avg_df,
        'explanations': explanations,
        'dominance_analysis': dominance_analysis,
        'df_scaled': df_scaled
    }
# Función para imprimir un resumen comprensible de los resultados
def print_correlation_analysis_summary(results):
    """
    Imprime un resumen legible de los resultados del análisis de correlación.
    """
    print("\n===== RESUMEN DEL ANÁLISIS DE CORRELACIÓN =====\n")
    
    print("CORRELACIONES PROMEDIO POR CATEGORÍA:")
    print(results['avg_correlations'])
    
    print("\nANÁLISIS POR TIPO DE PUNTUACIÓN:")
    for explanation in results['explanations']:
        print(f"\n• {explanation}")
    
    print("\nANÁLISIS DE DOMINANCIA GENERAL:")
    print(results['dominance_analysis'])
    
    print("\nINTERPRETACIÓN:")
    if 'Superficie' in results['avg_correlations'].columns and results['avg_correlations'].mean()['Superficie'] > results['avg_correlations'].mean()['Estructura']:
        print("Los resultados sugieren que las evaluaciones están más influenciadas por características superficiales")
        print("(longitud de palabras, legibilidad, uso de conectores) que por la estructura lingüística más profunda.")
        print("Esto apoya la hipótesis de que la forma puede tener más peso que el contenido en las evaluaciones.")
    else:
        print("Los resultados sugieren que las características estructurales del lenguaje tienen")
        print("mayor influencia en las evaluaciones que las características superficiales.")
        print("Esto no apoya completamente la hipótesis de que la forma domina sobre el contenido.")
    
    print("\nNOTA IMPORTANTE:")
    print("Este análisis se limita a características lingüísticas cuantificables y no mide")
    print("aspectos semánticos profundos como precisión factual, coherencia lógica o relevancia.")
    print("Un análisis completo de 'forma vs. contenido' requeriría métricas adicionales de calidad de contenido.")



def prepare_dataframe_for_analysis(df, text_column='response'):
    """
    Prepara el dataframe extrayendo todas las características lingüísticas necesarias.
    
    Args:
        df: DataFrame con las respuestas y puntuaciones
        text_column: Nombre de la columna que contiene el texto a analizar
        
    Returns:
        DataFrame con características lingüísticas añadidas
    """
    import pandas as pd
    import numpy as np
    from tqdm import tqdm  # Para mostrar una barra de progreso
    
    print(f"Extrayendo características lingüísticas de {len(df)} respuestas...")
    
    # Crear una copia para no modificar el original
    processed_df = df.copy()
    
    # Extraer características lingüísticas con barra de progreso
    features_list = []
    for i, text in tqdm(enumerate(processed_df[text_column]), total=len(processed_df)):
        try:
            features = extract_linguistic_features(text)
            features_list.append(features)
        except Exception as e:
            print(f"Error procesando fila {i}: {e}")
            # Proporcionar un diccionario de características vacío en caso de error
            features_list.append({
                'word_count': 0, 'sentence_count': 0, 'avg_word_length': 0,
                'avg_sentence_length': 0, 'unique_words_ratio': 0, 'lexical_diversity': 0,
                'reading_ease': 0, 'grade_level': 0, 'verb_ratio': 0, 'noun_ratio': 0,
                'adj_ratio': 0, 'adv_ratio': 0, 'pronoun_ratio': 0, 'named_entity_count': 0,
                'dependency_distance': 0, 'connector_count': 0
            })
    
    # Convertir la lista de diccionarios en columnas
    for feature in features_list[0].keys():
        processed_df[feature] = [features.get(feature, 0) for features in features_list]
    
    print("Extracción de características completada.")
    return processed_df

def run_complete_correlation_analysis(df, text_column='response', save_figures=True):
    """
    Ejecuta un análisis completo de correlación entre características lingüísticas escaladas y puntuaciones.
    
    Args:
        df: DataFrame con las respuestas y puntuaciones
        text_column: Nombre de la columna que contiene el texto a analizar
        save_figures: Si se deben guardar las figuras en archivos
        
    Returns:
        Dict con resultados y figuras
    """
    # 1. Preparar el dataframe añadiendo características lingüísticas
    print("Paso 1: Extrayendo características lingüísticas...")
    df_with_features = prepare_dataframe_for_analysis(df, text_column=text_column)
    
    # 2. Crear heatmap y análisis de correlación con valores escalados
    print("Paso 2: Escalando características y generando matrices de correlación...")
    correlation_results = create_correlation_heatmap_scaled(df_with_features)
    
    if correlation_results is None:
        print("Error al generar el análisis de correlación.")
        return None
    
    # 3. Identificar las características más predictivas para cada puntuación
    print("Paso 3: Analizando características más predictivas por tipo de puntuación...")
    score_columns = [col for col in df.columns if col.endswith('_score')]
    
    # 4. Imprimir resumen de hallazgos
    print("\n===== RESUMEN DEL ANÁLISIS DE CORRELACIÓN (CARACTERÍSTICAS ESCALADAS) =====\n")
    
    # Mostrar la tabla de correlaciones promedio
    print("CORRELACIONES PROMEDIO POR CATEGORÍA:")
    print(correlation_results['avg_correlations'])
    
    # Mostrar análisis por tipo de puntuación
    print("\nANÁLISIS POR TIPO DE PUNTUACIÓN:")
    for explanation in correlation_results['explanations']:
        print(f"\n• {explanation}")
    
    # Mostrar análisis de dominancia general
    print("\nANÁLISIS DE DOMINANCIA GENERAL:")
    print(correlation_results['dominance_analysis'])
    
    # Interpretación de los resultados
    print("\nINTERPRETACIÓN:")
    avg_corrs = correlation_results['avg_correlations'].mean()
    
    if 'Superficie' in avg_corrs.index and 'Estructura' in avg_corrs.index:
        if avg_corrs['Superficie'] > avg_corrs['Estructura']:
            print("Los resultados con características escaladas sugieren que las evaluaciones están más")
            print("influenciadas por características superficiales (longitud de palabras, legibilidad,")
            print("uso de conectores) que por la estructura lingüística más profunda.")
            print("Esto apoya la hipótesis de que la forma puede tener más peso que el contenido.")
        else:
            print("Los resultados con características escaladas sugieren que las características")
            print("estructurales del lenguaje tienen mayor influencia en las evaluaciones que las")
            print("características superficiales. Esto no apoya completamente la hipótesis de que")
            print("la forma domina sobre el contenido.")
    
    print("\nNOTA SOBRE EL ESCALADO:")
    print("Al escalar todas las características entre 0 y 1, hemos eliminado las diferencias en")
    print("magnitud y unidades entre las distintas características, lo que permite una comparación")
    print("más equitativa de la importancia relativa de cada característica en las puntuaciones.")
    
    # 5. Guardar figuras si se solicitó
    if save_figures:
        correlation_results['heatmap_fig'].savefig('correlation_heatmap_scaled.png', dpi=300, bbox_inches='tight')
        correlation_results['avg_heatmap_fig'].savefig('avg_correlation_by_category_scaled.png', dpi=300, bbox_inches='tight')
        correlation_results['bar_fig'].savefig('correlation_by_score_category_scaled.png', dpi=300, bbox_inches='tight')
        print("\nFiguras guardadas como:")
        print("- correlation_heatmap_scaled.png")
        print("- avg_correlation_by_category_scaled.png")
        print("- correlation_by_score_category_scaled.png")
    
    return {
        'df_with_features': df_with_features,
        'df_scaled': correlation_results['df_scaled'],
        'correlation_results': correlation_results,
        'avg_correlations': correlation_results['avg_correlations']
    }

In [ ]:
results = run_complete_correlation_analysis(evaluations_with_answers_ia, text_column='response')

# Mostrar los gráficos interactivamente
plt.show()

In [ ]:
# Cargar los datos
answers = load_data("answers")
evaluations = load_data("eval")

# Convertir los IDs a string para asegurar compatibilidad
answers["_id"] = answers["_id"].astype(str)
evaluations["response"] = evaluations["original_answer"]
evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)

# Método 1: Usando merge con todas las columnas necesarias
evaluations_with_answers = evaluations.merge(
    answers[['_id', 'answer', 'bibliography_sources', 'confidence_level']], 
    left_on='original_answer_id',
    right_on='_id',
    how='left'
)

# Función para combinar respuesta y bibliografía cuando esté disponible
def combine_answer_and_bibliography(row):
    if pd.notna(row['bibliography_sources']):
        return row['answer'] + '\n\nReferences: ' + row['bibliography_sources']
    else:
        return row['answer']

# Crear la columna original_answer con la combinación de respuesta y bibliografía
evaluations_with_answers['original_answer'] = evaluations_with_answers.apply(combine_answer_and_bibliography, axis=1)


# Verificar el resultado
print("Primeras filas del DataFrame final:")
print(evaluations_with_answers.head())
evaluations_with_answers["response"] = evaluations_with_answers["original_answer"]
evaluations_with_answers, _ , _ = filter_and_analyze_responses(evaluations_with_answers)

In [ ]:
results = run_linguistic_analysis(evaluations_with_answers)

In [ ]:
results = run_complete_correlation_analysis(evaluations_with_answers, text_column='response')

# Mostrar los gráficos interactivamente
plt.show()

In [ ]:
# Cargar los datos
answers = load_data("answers")
evaluations = load_data("ia_deepseek_eval")

# Convertir los IDs a string para asegurar compatibilidad
answers["_id"] = answers["_id"].astype(str)
evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)

# Método 1: Usando merge con todas las columnas necesarias
evaluations_with_answers = evaluations.merge(
    answers[['_id', 'answer', 'bibliography_sources']], 
    left_on='original_answer_id',
    right_on='_id',
    how='left'
)

# Función para combinar respuesta y bibliografía cuando esté disponible
def combine_answer_and_bibliography(row):
    if pd.notna(row['bibliography_sources']):
        return row['answer'] + '\n\nReferences: ' + row['bibliography_sources']
    else:
        return row['answer']

# Crear la columna original_answer con la combinación de respuesta y bibliografía
evaluations_with_answers['original_answer'] = evaluations_with_answers.apply(combine_answer_and_bibliography, axis=1)


# Verificar el resultado
print("Primeras filas del DataFrame final:")
print(evaluations_with_answers.head())
evaluations_with_answers["response"] = evaluations_with_answers["original_answer"]
evaluations_with_answers, _ , _ = filter_and_analyze_responses(evaluations_with_answers)

In [ ]:
results = run_complete_correlation_analysis(evaluations_with_answers, text_column='response')

# Mostrar los gráficos interactivamente
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler

def create_correlation_heatmap_multi_df(dataframes, df_names, text_column='response'):
    """
    Crea un heatmap de correlaciones con múltiples dataframes, analizando cada uno por separado.
    
    Args:
        dataframes: Lista de DataFrames con respuestas y puntuaciones
        df_names: Lista de nombres para identificar cada dataframe
        text_column: Nombre de la columna que contiene el texto a analizar
        
    Returns:
        Dict con resultados y visualizaciones
    """
    # Verificar que los parámetros sean correctos
    if len(dataframes) != len(df_names):
        raise ValueError("El número de dataframes debe ser igual al número de nombres proporcionados")
    
    # Preparar cada dataframe y extraer características
    processed_dfs = []
    for i, df in enumerate(dataframes):
        print(f"Procesando dataframe {df_names[i]}...")
        processed_df = prepare_dataframe_for_analysis(df, text_column=text_column)
        processed_dfs.append(processed_df)
    
    # Definir las categorías de características
    count_features = ['word_count', 'sentence_count']
    
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Todas las características
    all_features = count_features + surface_features + structure_features
    
    # Analizar cada dataframe por separado
    all_correlations = []
    prefixed_score_columns = []
    
    for i, df in enumerate(processed_dfs):
        prefix = df_names[i]
        
        # Verificar qué características están disponibles
        available_features = [f for f in all_features if f in df.columns]
        
        # Verificar qué puntuaciones están disponibles
        score_columns = ["accuracy_score", "clarity_score", "completeness_score"]
        available_scores = [s for s in score_columns if s in df.columns]
        
        if not available_features or not available_scores:
            print(f"No se encontraron suficientes datos en {prefix}. Omitiendo...")
            continue
        
        # Escalar características
        df_scaled = df.copy()
        scaler = MinMaxScaler()
        df_scaled[available_features] = scaler.fit_transform(df[available_features])
        
        # Calcular correlaciones
        corr_matrix = df_scaled[available_features + available_scores].corr()
        feature_score_corr = corr_matrix.loc[available_features, available_scores]
        
        # Guardar con prefijos para cada score
        for score in available_scores:
            prefixed_score = f"{prefix}_{score}"
            prefixed_score_columns.append(prefixed_score)
            
            # Guardar correlaciones para esta puntuación
            all_correlations.append(
                pd.DataFrame(
                    feature_score_corr[score].values, 
                    index=available_features,
                    columns=[prefixed_score]
                )
            )
    
    # Combinar todas las correlaciones
    if not all_correlations:
        print("No se pudieron calcular correlaciones para ningún dataframe.")
        return None
    
    combined_corr = pd.concat(all_correlations, axis=1)
    
    # Verificar características disponibles en el resultado combinado
    available_features = combined_corr.index.tolist()
    
    # Ordenar características por categoría para visualización
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        ordered_features.extend([f for f in feature_group if f in available_features])
    
    # Crear figura para el heatmap principal
    plt.figure(figsize=(len(prefixed_score_columns) * 1.5, len(ordered_features) * 0.5))
    
    # Reordenar filas para agrupar por categoría
    sorted_corr = combined_corr.loc[ordered_features, :]
    
    # Crear heatmap con anotaciones
    heatmap = sns.heatmap(
        sorted_corr, 
        annot=True,
        cmap='coolwarm',
        vmin=-1,
        vmax=1,
        center=0,
        fmt='.2f',
        linewidths=.5,
    )
    
    # Añadir líneas para separar categorías
    count_end = sum(1 for feature in count_features if feature in ordered_features)
    surface_end = count_end + sum(1 for feature in surface_features if feature in ordered_features)
    
    plt.axhline(y=count_end, color='black', linewidth=2)
    plt.axhline(y=surface_end, color='black', linewidth=2)
    
    # Añadir etiquetas de categoría
    plt.text(-2.5, count_end / 2, 'CONTEO', verticalalignment='center', fontsize=12, fontweight='bold')
    plt.text(-2.5, count_end + (surface_end - count_end) / 2, 'SUPERFICIE', verticalalignment='center', fontsize=12, fontweight='bold')
    plt.text(-2.5, surface_end + (len(ordered_features) - surface_end) / 2, 'ESTRUCTURA', verticalalignment='center', fontsize=12, fontweight='bold')
    
    plt.title('Correlaciones entre Características Lingüísticas y Puntuaciones por Fuente', fontsize=16)
    plt.tight_layout()
 
    
    # Calcular correlaciones promedio por categoría y fuente
    avg_correlations = {}
    
    for prefix in df_names:
        prefix_scores = [col for col in prefixed_score_columns if col.startswith(prefix)]
        if not prefix_scores:
            continue
            
        # Calcular promedios por categoría
        avg_by_category = {}
        
        # Conteo
        count_features_available = [f for f in count_features if f in available_features]
        if count_features_available:
            count_correlations = sorted_corr.loc[count_features_available, prefix_scores].abs().mean().mean()
            avg_by_category['Conteo'] = count_correlations
        
        # Superficie
        surface_features_available = [f for f in surface_features if f in available_features]
        if surface_features_available:
            surface_correlations = sorted_corr.loc[surface_features_available, prefix_scores].abs().mean().mean()
            avg_by_category['Superficie'] = surface_correlations
        
        # Estructura
        structure_features_available = [f for f in structure_features if f in available_features]
        if structure_features_available:
            structure_correlations = sorted_corr.loc[structure_features_available, prefix_scores].abs().mean().mean()
            avg_by_category['Estructura'] = structure_correlations
        
        avg_correlations[prefix] = avg_by_category
    
    # Crear dataframe para visualización comparativa
    comparison_data = []
    for source, categories in avg_correlations.items():
        for category, value in categories.items():
            comparison_data.append({
                'Fuente': source,
                'Categoría': category,
                'Correlación Promedio': value
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Crear heatmap de comparación
    if len(comparison_df) > 0:
        fig_comparison, ax_comparison = plt.subplots(figsize=(8, 6))
        
        # Crear pivot para el heatmap
        pivot_df = comparison_df.pivot(index='Categoría', columns='Fuente', values='Correlación Promedio')
        
        sns.heatmap(pivot_df, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.3f', linewidths=.5, ax=ax_comparison)
        ax_comparison.set_title('Comparación de Correlaciones Promedio por Fuente', fontsize=14)
    else:
        fig_comparison = None
    
    # Crear gráfico de barras para comparación
    if len(comparison_df) > 0:
        fig_bar, ax_bar = plt.subplots(figsize=(10, 6))
        
        sns.barplot(x='Categoría', y='Correlación Promedio', hue='Fuente', data=comparison_df, ax=ax_bar)
        ax_bar.set_title('Correlación Promedio por Categoría y Fuente', fontsize=14)
        ax_bar.set_ylabel('Correlación Promedio (valor absoluto)')
        ax_bar.set_xlabel('Categoría de Características')
        plt.legend(title='Fuente')
        plt.tight_layout()
    else:
        fig_bar = None
    
    # Crear análisis comparativo
    comparative_analysis = []
    
    for category in ['Conteo', 'Superficie', 'Estructura']:
        category_data = comparison_df[comparison_df['Categoría'] == category]
        if len(category_data) < 2:
            continue
            
        sources = category_data['Fuente'].tolist()
        values = category_data['Correlación Promedio'].tolist()
        
        if len(sources) >= 2:
            max_idx = values.index(max(values))
            min_idx = values.index(min(values))
            
            analysis = f"Para la categoría {category}, {sources[max_idx]} muestra la mayor correlación "
            analysis += f"({values[max_idx]:.3f}), mientras que {sources[min_idx]} muestra la menor "
            analysis += f"({values[min_idx]:.3f}). "
            
            diff_pct = ((values[max_idx] / values[min_idx]) - 1) * 100 if values[min_idx] > 0 else 0
            if diff_pct > 10:
                analysis += f"Esto representa una diferencia del {diff_pct:.1f}%, lo que sugiere una "
                analysis += f"sensibilidad considerablemente distinta a las características de {category} "
                analysis += f"entre estas fuentes de evaluación."
            
            comparative_analysis.append(analysis)
    
    # Análisis general
    general_analysis = "Análisis general por fuente:\n"
    
    for source in df_names:
        if source in avg_correlations:
            categories = avg_correlations[source]
            if categories:
                max_category = max(categories.items(), key=lambda x: x[1])
                
                general_analysis += f"- {source}: La categoría dominante es {max_category[0]} ({max_category[1]:.3f}), "
                
                other_categories = {k: v for k, v in categories.items() if k != max_category[0]}
                if other_categories:
                    other_items = list(other_categories.items())
                    
                    for i, (cat, val) in enumerate(other_items):
                        diff = max_category[1] - val
                        general_analysis += f"superando a {cat} por {diff:.3f} ({val:.3f})"
                        
                        if i < len(other_items) - 1:
                            general_analysis += " y "
                
                general_analysis += ".\n"
    
    return {
        'heatmap_fig': plt.gcf(),
        'comparison_fig': fig_comparison,
        'bar_fig': fig_bar,
        'correlation_matrix': sorted_corr,
        'avg_correlations': avg_correlations,
        'comparison_df': comparison_df,
        'comparative_analysis': comparative_analysis,
        'general_analysis': general_analysis
    }

def run_multi_df_correlation_analysis(dataframes, df_names, text_column='response', save_figures=True):
    """
    Ejecuta un análisis completo de correlación comparando múltiples dataframes.
    
    Args:
        dataframes: Lista de DataFrames con las respuestas y puntuaciones
        df_names: Lista de nombres para identificar cada dataframe
        text_column: Nombre de la columna que contiene el texto a analizar
        save_figures: Si se deben guardar las figuras en archivos
        
    Returns:
        Dict con resultados y figuras
    """
    print(f"Iniciando análisis comparativo entre {len(dataframes)} fuentes de evaluación...")
    
    # Ejecutar el análisis comparativo
    results = create_correlation_heatmap_multi_df(dataframes, df_names, text_column)
    
    if results is None:
        print("Error al generar el análisis de correlación.")
        return None
    
    # Imprimir resumen de hallazgos
    print("\n===== RESUMEN DEL ANÁLISIS COMPARATIVO =====\n")
    
    # Mostrar comparación general entre fuentes
    if 'comparison_df' in results and len(results['comparison_df']) > 0:
        pivot_df = results['comparison_df'].pivot(index='Categoría', columns='Fuente', values='Correlación Promedio')
        print("CORRELACIONES PROMEDIO POR CATEGORÍA Y FUENTE:")
        print(pivot_df)
    
    # Análisis comparativo entre fuentes
    if 'comparative_analysis' in results and results['comparative_analysis']:
        print("\nANÁLISIS COMPARATIVO ENTRE FUENTES:")
        for analysis in results['comparative_analysis']:
            print(f"\n• {analysis}")
    
    # Análisis general
    if 'general_analysis' in results:
        print("\nANÁLISIS GENERAL POR FUENTE:")
        print(results['general_analysis'])
    
    # Guardar figuras si se solicitó
    if save_figures:
        if 'heatmap_fig' in results and results['heatmap_fig'] is not None:
            results['heatmap_fig'].savefig('multi_correlation_heatmap.png', dpi=300, bbox_inches='tight')
            print("\nFigura guardada como: multi_correlation_heatmap.png")
            
        if 'comparison_fig' in results and results['comparison_fig'] is not None:
            results['comparison_fig'].savefig('comparison_heatmap.png', dpi=300, bbox_inches='tight')
            print("Figura guardada como: comparison_heatmap.png")
            
        if 'bar_fig' in results and results['bar_fig'] is not None:
            results['bar_fig'].savefig('comparison_barplot.png', dpi=300, bbox_inches='tight')
            print("Figura guardada como: comparison_barplot.png")
    
    return results

# Ejemplo de uso:
"""
# Para ejecutar el análisis comparativo:
results = run_multi_df_correlation_analysis(
    dataframes=[df_ai, df_human],
    df_names=['AI', 'Human'],
    text_column='response'
)

# Para mostrar los gráficos:
import matplotlib.pyplot as plt
plt.show()
"""

In [ ]:
# Cargar los datos
answers = load_data("answers")
evaluations = load_data("ia_eval")

# Convertir los IDs a string para asegurar compatibilidad
answers["_id"] = answers["_id"].astype(str)
evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)

# Método 1: Usando merge con todas las columnas necesarias
evaluations_with_answers_ia = evaluations.merge(
    answers[['_id', 'answer', 'bibliography_sources', 'confidence_level']], 
    left_on='original_answer_id',
    right_on='_id',
    how='left'
)

# Función para combinar respuesta y bibliografía cuando esté disponible
def combine_answer_and_bibliography(row):
    if pd.notna(row['bibliography_sources']):
        return row['answer'] + '\n\nReferences: ' + row['bibliography_sources']
    else:
        return row['answer']

# Crear la columna original_answer con la combinación de respuesta y bibliografía
evaluations_with_answers_ia['original_answer'] = evaluations_with_answers_ia.apply(combine_answer_and_bibliography, axis=1)


# Verificar el resultado
print("Primeras filas del DataFrame final:")
print(evaluations_with_answers_ia.head())
evaluations_with_answers_ia["response"] = evaluations_with_answers_ia["original_answer"]
evaluations_with_answers_ia, _ , _ = filter_and_analyze_responses(evaluations_with_answers_ia)


# Cargar los datos
answers = load_data("answers")
evaluations = load_data("eval")

# Convertir los IDs a string para asegurar compatibilidad
answers["_id"] = answers["_id"].astype(str)
evaluations["response"] = evaluations["original_answer"]
evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)

# Método 1: Usando merge con todas las columnas necesarias
evaluations_with_answers = evaluations.merge(
    answers[['_id', 'answer', 'bibliography_sources', 'confidence_level']], 
    left_on='original_answer_id',
    right_on='_id',
    how='left'
)

# Función para combinar respuesta y bibliografía cuando esté disponible
def combine_answer_and_bibliography(row):
    if pd.notna(row['bibliography_sources']):
        return row['answer'] + '\n\nReferences: ' + row['bibliography_sources']
    else:
        return row['answer']

# Crear la columna original_answer con la combinación de respuesta y bibliografía
evaluations_with_answers['original_answer'] = evaluations_with_answers.apply(combine_answer_and_bibliography, axis=1)


# Verificar el resultado
print("Primeras filas del DataFrame final:")
print(evaluations_with_answers.head())
evaluations_with_answers["response"] = evaluations_with_answers["original_answer"]
evaluations_with_answers, _ , _ = filter_and_analyze_responses(evaluations_with_answers)

In [ ]:
# Para ejecutar el análisis comparativo:
df_human = evaluations_with_answers
df_ai = evaluations_with_answers_ia

results = run_multi_df_correlation_analysis(
    dataframes=[df_ai, df_human],
    df_names=['AI', 'human'],
    text_column='response'
)

# Para mostrar los gráficos:
import matplotlib.pyplot as plt
plt.show()

In [ ]:
def create_correlation_heatmap_by_llm(dataframes, df_names, text_column='response', llm_column='LLM'):
    """
    Crea heatmaps de correlaciones separados por tipo de LLM (0=humano, 1=IA, 2=IA CoT)
    para cada conjunto de datos en dataframes.
    
    Args:
        dataframes: Lista de DataFrames con respuestas y puntuaciones
        df_names: Lista de nombres para identificar cada dataframe
        text_column: Nombre de la columna que contiene el texto a analizar
        llm_column: Nombre de la columna que contiene el tipo de LLM
        
    Returns:
        Dict con resultados y visualizaciones separados por tipo de LLM
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from tqdm import tqdm
    from sklearn.preprocessing import MinMaxScaler
    
    # Verificar que los parámetros sean correctos
    if len(dataframes) != len(df_names):
        raise ValueError("El número de dataframes debe ser igual al número de nombres proporcionados")
    
    # Preparar cada dataframe y extraer características
    processed_dfs = []
    for i, df in enumerate(dataframes):
        print(f"Procesando dataframe {df_names[i]}...")
        processed_df = prepare_dataframe_for_analysis(df, text_column=text_column)
        processed_dfs.append(processed_df)
    
    # Definir las categorías de características
    count_features = ['word_count', 'sentence_count']
    
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Todas las características
    all_features = count_features + surface_features + structure_features
    
    # Definir tipos de LLM
    llm_types = {0: "Humano", 1: "IA", 2: "IA CoT"}
    
    # Resultados para cada tipo de LLM
    results_by_llm = {}
    
    # Analizar cada tipo de LLM por separado
    for llm_value, llm_name in llm_types.items():
        print(f"\nAnalizando correlaciones para respuestas de {llm_name} (LLM={llm_value})...")
        
        # Filtrar dataframes por tipo de LLM
        filtered_dfs = []
        filtered_df_names = []
        
        for i, df in enumerate(processed_dfs):
            if llm_column in df.columns:
                filtered_df = df[df[llm_column] == llm_value].copy()
                
                if len(filtered_df) > 0:
                    filtered_dfs.append(filtered_df)
                    filtered_df_names.append(df_names[i])
                    print(f"  - {df_names[i]}: {len(filtered_df)} evaluaciones encontradas")
                else:
                    print(f"  - {df_names[i]}: No hay evaluaciones para LLM={llm_value}")
            else:
                print(f"  - {df_names[i]}: No contiene la columna '{llm_column}'")
        
        if not filtered_dfs:
            print(f"No se encontraron datos para LLM={llm_value}")
            continue
        
        # Analizar correlaciones para este tipo de LLM
        all_correlations = []
        prefixed_score_columns = []
        
        for i, df in enumerate(filtered_dfs):
            prefix = filtered_df_names[i]
            
            # Verificar qué características están disponibles
            available_features = [f for f in all_features if f in df.columns]
            
            # Verificar qué puntuaciones están disponibles
            score_columns = ["accuracy_score", "clarity_score", "completeness_score"]
            available_scores = [s for s in score_columns if s in df.columns]
            
            if not available_features or not available_scores:
                print(f"No se encontraron suficientes datos en {prefix}. Omitiendo...")
                continue
            
            # Escalar características
            df_scaled = df.copy()
            scaler = MinMaxScaler()
            df_scaled[available_features] = scaler.fit_transform(df[available_features])
            
            # Calcular correlaciones
            corr_matrix = df_scaled[available_features + available_scores].corr()
            feature_score_corr = corr_matrix.loc[available_features, available_scores]
            
            # Guardar con prefijos para cada score
            for score in available_scores:
                prefixed_score = f"{prefix}_{score}"
                prefixed_score_columns.append(prefixed_score)
                
                # Guardar correlaciones para esta puntuación
                all_correlations.append(
                    pd.DataFrame(
                        feature_score_corr[score].values, 
                        index=available_features,
                        columns=[prefixed_score]
                    )
                )
        
        # Combinar todas las correlaciones
        if not all_correlations:
            print(f"No se pudieron calcular correlaciones para LLM={llm_value}")
            continue
        
        combined_corr = pd.concat(all_correlations, axis=1)
        
        # Verificar características disponibles en el resultado combinado
        available_features = combined_corr.index.tolist()
        
        # Ordenar características por categoría para visualización
        ordered_features = []
        for feature_group in [count_features, surface_features, structure_features]:
            ordered_features.extend([f for f in feature_group if f in available_features])
        
        # Crear figura para el heatmap principal
        plt.figure(figsize=(20, len(ordered_features) * 0.5))  # Ancho aumentado
        
        # Reordenar filas para agrupar por categoría
        sorted_corr = combined_corr.loc[ordered_features, :]
        
        # Crear heatmap con anotaciones
        heatmap = sns.heatmap(
            sorted_corr, 
            annot=True,
            cmap='coolwarm',
            vmin=-1,
            vmax=1,
            center=0,
            fmt='.2f',
            linewidths=.5,
        )
        
        # Añadir líneas para separar categorías
        count_end = sum(1 for feature in count_features if feature in ordered_features)
        surface_end = count_end + sum(1 for feature in surface_features if feature in ordered_features)
        
        plt.axhline(y=count_end, color='black', linewidth=2)
        plt.axhline(y=surface_end, color='black', linewidth=2)
        
        # Añadir etiquetas de categoría - ajustadas para un gráfico más ancho
        plt.text(-1.0, count_end / 2, 'CONTEO', verticalalignment='center', fontsize=12, fontweight='bold')
        plt.text(-1.0, count_end + (surface_end - count_end) / 2, 'SUPERFICIE', verticalalignment='center', fontsize=12, fontweight='bold')
        plt.text(-1.0, surface_end + (len(ordered_features) - surface_end) / 2, 'ESTRUCTURA', verticalalignment='center', fontsize=12, fontweight='bold')
        
        plt.title(f'Correlaciones para Respuestas de {llm_name} (LLM={llm_value})', fontsize=16)
        plt.tight_layout()
     
        
        # Calcular correlaciones promedio por categoría y fuente
        avg_correlations = {}
        
        for prefix in filtered_df_names:
            prefix_scores = [col for col in prefixed_score_columns if col.startswith(prefix)]
            if not prefix_scores:
                continue
                
            # Calcular promedios por categoría
            avg_by_category = {}
            
            # Conteo
            count_features_available = [f for f in count_features if f in available_features]
            if count_features_available:
                count_correlations = sorted_corr.loc[count_features_available, prefix_scores].abs().mean().mean()
                avg_by_category['Conteo'] = count_correlations
            
            # Superficie
            surface_features_available = [f for f in surface_features if f in available_features]
            if surface_features_available:
                surface_correlations = sorted_corr.loc[surface_features_available, prefix_scores].abs().mean().mean()
                avg_by_category['Superficie'] = surface_correlations
            
            # Estructura
            structure_features_available = [f for f in structure_features if f in available_features]
            if structure_features_available:
                structure_correlations = sorted_corr.loc[structure_features_available, prefix_scores].abs().mean().mean()
                avg_by_category['Estructura'] = structure_correlations
            
            avg_correlations[prefix] = avg_by_category
        
        # Crear dataframe para visualización comparativa
        comparison_data = []
        for source, categories in avg_correlations.items():
            for category, value in categories.items():
                comparison_data.append({
                    'Fuente': source,
                    'Categoría': category,
                    'Correlación Promedio': value
                })
        
        comparison_df = pd.DataFrame(comparison_data)
        
        # Crear heatmap de comparación
        if len(comparison_df) > 0:
            fig_comparison, ax_comparison = plt.subplots(figsize=(8, 6))
            
            # Crear pivot para el heatmap
            pivot_df = comparison_df.pivot(index='Categoría', columns='Fuente', values='Correlación Promedio')
            
            sns.heatmap(pivot_df, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.3f', linewidths=.5, ax=ax_comparison)
            ax_comparison.set_title(f'Comparación para {llm_name} (LLM={llm_value})', fontsize=14)
        else:
            fig_comparison = None
        
        # Crear gráfico de barras para comparación
        if len(comparison_df) > 0:
            fig_bar, ax_bar = plt.subplots(figsize=(10, 6))
            
            sns.barplot(x='Categoría', y='Correlación Promedio', hue='Fuente', data=comparison_df, ax=ax_bar)
            ax_bar.set_title(f'Correlación Promedio para {llm_name} (LLM={llm_value})', fontsize=14)
            ax_bar.set_ylabel('Correlación Promedio (valor absoluto)')
            ax_bar.set_xlabel('Categoría de Características')
            plt.legend(title='Fuente')
            plt.tight_layout()
        else:
            fig_bar = None
        
        # Guardar resultados para este tipo de LLM
        results_by_llm[llm_value] = {
            'llm_name': llm_name,
            'heatmap_fig': plt.gcf(),
            'comparison_fig': fig_comparison,
            'bar_fig': fig_bar,
            'correlation_matrix': sorted_corr,
            'avg_correlations': avg_correlations,
            'comparison_df': comparison_df
        }
    
    return results_by_llm

def run_llm_type_correlation_analysis(dataframes, df_names, text_column='response', llm_column='LLM', save_figures=True):
    """
    Ejecuta un análisis completo de correlación separado por tipo de LLM (0=humano, 1=IA, 2=IA CoT).
    
    Args:
        dataframes: Lista de DataFrames con las respuestas y puntuaciones
        df_names: Lista de nombres para identificar cada dataframe
        text_column: Nombre de la columna que contiene el texto a analizar
        llm_column: Nombre de la columna que identifica el tipo de LLM
        save_figures: Si se deben guardar las figuras en archivos
        
    Returns:
        Dict con resultados y figuras para cada tipo de LLM
    """
    print(f"Iniciando análisis por tipo de LLM entre {len(dataframes)} fuentes de evaluación...")
    
    # Ejecutar el análisis comparativo
    results_by_llm = create_correlation_heatmap_by_llm(dataframes, df_names, text_column, llm_column)
    
    if not results_by_llm:
        print("Error al generar el análisis de correlación por tipo de LLM.")
        return None
    
    # Imprimir resumen de hallazgos para cada tipo de LLM
    for llm_value, results in results_by_llm.items():
        llm_name = results['llm_name']
        print(f"\n===== RESUMEN DEL ANÁLISIS PARA {llm_name.upper()} (LLM={llm_value}) =====\n")
        
        # Mostrar comparación general entre fuentes
        if 'comparison_df' in results and len(results['comparison_df']) > 0:
            pivot_df = results['comparison_df'].pivot(index='Categoría', columns='Fuente', values='Correlación Promedio')
            print("CORRELACIONES PROMEDIO POR CATEGORÍA Y FUENTE:")
            print(pivot_df)
        
        # Guardar figuras si se solicitó
        if save_figures:
            if 'heatmap_fig' in results and results['heatmap_fig'] is not None:
                filename = f'heatmap_llm{llm_value}_{results["llm_name"]}.png'
                results['heatmap_fig'].savefig(filename, dpi=300, bbox_inches='tight')
                print(f"Figura guardada como: {filename}")
                
            if 'comparison_fig' in results and results['comparison_fig'] is not None:
                filename = f'comparison_heatmap_llm{llm_value}_{results["llm_name"]}.png'
                results['comparison_fig'].savefig(filename, dpi=300, bbox_inches='tight')
                print(f"Figura guardada como: {filename}")
                
            if 'bar_fig' in results and results['bar_fig'] is not None:
                filename = f'barplot_llm{llm_value}_{results["llm_name"]}.png'
                results['bar_fig'].savefig(filename, dpi=300, bbox_inches='tight')
                print(f"Figura guardada como: {filename}")
    
    # Crear un análisis comparativo entre tipos de LLM
    print("\n===== COMPARACIÓN ENTRE TIPOS DE LLM =====\n")
    
    # Comparar categorías dominantes por fuente y tipo de LLM
    all_dominants = {}
    
    for llm_value, results in results_by_llm.items():
        llm_name = results['llm_name']
        for source, categories in results['avg_correlations'].items():
            if not categories:
                continue
                
            dominant_category = max(categories.items(), key=lambda x: x[1])
            
            if source not in all_dominants:
                all_dominants[source] = {}
            
            all_dominants[source][llm_name] = {
                'dominant_category': dominant_category[0],
                'correlation': dominant_category[1]
            }
    
    # Imprimir comparación
    for source, llm_data in all_dominants.items():
        print(f"\nFuente: {source}")
        for llm_name, data in llm_data.items():
            print(f"  - {llm_name}: {data['dominant_category']} ({data['correlation']:.3f})")
    
    return results_by_llm



In [ ]:
# Para ejecutar el análisis por tipo de LLM:
results_by_llm = run_llm_type_correlation_analysis(
    dataframes=[df_ai, df_human],
    df_names=['AI', 'Human'],
    text_column='response',
    llm_column='LLM'
)

# Para mostrar los gráficos:
import matplotlib.pyplot as plt
plt.show()

In [ ]:
def create_comparative_correlation_heatmap_with_bootstrap(df_ai, df_human, text_column='response', llm_column='LLM', n_bootstrap=1000, n_samples=50, confidence=0.95):
    """
    Creates a correlation heatmap with confidence intervals calculated through bootstrapping using Plotly.
    
    Args:
        df_ai: DataFrame with AI evaluations
        df_human: DataFrame with human evaluations
        text_column: Name of the column containing the text to analyze
        llm_column: Name of the column containing the LLM type
        n_bootstrap: Number of bootstrap samples for confidence intervals
        confidence: Confidence level for intervals (default 0.95)
        
    Returns:
        Plotly figure with comparative heatmap and DataFrame with correlations and intervals
    """
    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go
    from scipy.stats import spearmanr
    from tqdm import tqdm
    
    def bootstrap_correlation(data, features, target, sample_size=50, n_iterations=1000, conf_level=0.95):
        """
        Calculates correlations using bootstrap resampling technique.
        Takes 'sample_size' random samples, repeats 'n_iterations' times and averages the results.
        
        Args:
            data: DataFrame with data
            features: List of features to correlate
            target: Target variable (e.g., 'accuracy_score')
            sample_size: Size of each subsample (default 50)
            n_iterations: Number of iterations (default 1000)
            conf_level: Confidence level for intervals (default 0.95)
            
        Returns:
            Dict with correlations and their confidence intervals
        """
        import numpy as np
        import pandas as pd
        from scipy.stats import spearmanr
        
        # Verify that sample_size is not larger than available data
        n = len(data)
        if sample_size > n:
            print(f"WARNING: Sample size ({sample_size}) larger than available data ({n}). Adjusting to {n}.")
            sample_size = n
        
        correlations = []
        
        # Perform n_iterations of subsampling
        for _ in range(n_iterations):
            # Select sample_size random samples (with replacement)
            indices = np.random.choice(range(n), size=sample_size, replace=True)
            bootstrap_sample = data.iloc[indices]
            
            # Calculate correlations for this sample
            corr_sample = {}
            for feature in features:
                # Verify if there's enough valid data
                valid_data = ~(bootstrap_sample[feature].isna() | bootstrap_sample[target].isna())
                if valid_data.sum() < 3:  # Need at least 3 points for correlation
                    corr_sample[feature] = np.nan
                    continue
                    
                # Use scipy.stats.spearmanr to get only the coefficient
                corr, _ = spearmanr(bootstrap_sample.loc[valid_data, feature], 
                                bootstrap_sample.loc[valid_data, target])
                corr_sample[feature] = corr
            
            correlations.append(corr_sample)
        
        # Convert to DataFrame for easier calculation
        corr_df = pd.DataFrame(correlations)
        
        # Calculate point correlation and confidence intervals
        alpha = 1 - conf_level
        result = {}
        for feature in features:
            # Filter NaN values
            feature_corrs = corr_df[feature].dropna()
            
            if len(feature_corrs) == 0:
                # No valid correlations for this feature
                result[feature] = {
                    'correlation': np.nan,
                    'lower_ci': np.nan,
                    'upper_ci': np.nan,
                    'sample_count': 0
                }
                continue
            
            # Point estimate (bootstrap average)
            point_estimate = feature_corrs.mean()
            
            # Confidence intervals
            if len(feature_corrs) >= 3:  # Need at least 3 samples for CI
                lower_bound = np.percentile(feature_corrs, alpha/2 * 100)
                upper_bound = np.percentile(feature_corrs, (1 - alpha/2) * 100)
            else:
                # Not enough samples to calculate CIs
                lower_bound = np.nan
                upper_bound = np.nan
            
            result[feature] = {
                'correlation': point_estimate,
                'lower_ci': lower_bound,
                'upper_ci': upper_bound,
                'sample_count': len(feature_corrs)
            }
        
        return result
    
    # Prepare the dataframes
    print("Processing AI dataframe...")
    df_ai_processed = prepare_dataframe_for_analysis(df_ai, text_column=text_column)
    
    print("Processing Human dataframe...")
    df_human_processed = prepare_dataframe_for_analysis(df_human, text_column=text_column)
    
    # Define feature categories
    count_features = ['word_count', 'sentence_count']
    
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # All features
    all_features = count_features + surface_features + structure_features
    
    # Define LLM types
    llm_types = {0: "Human", 1: "AI", 2: "CoT"}
    
    # Check which features are available in both dataframes
    available_features = [f for f in all_features 
                         if f in df_ai_processed.columns and f in df_human_processed.columns]
    
    # Check if accuracy_score is available in both dataframes
    if "accuracy_score" not in df_ai_processed.columns:
        raise ValueError("'accuracy_score' not found in AI dataframe")
    if "accuracy_score" not in df_human_processed.columns:
        raise ValueError("'accuracy_score' not found in Human dataframe")
    
    # Create dataframes to store correlations and intervals
    point_correlations = pd.DataFrame(index=available_features)
    lower_ci = pd.DataFrame(index=available_features)
    upper_ci = pd.DataFrame(index=available_features)
    
    # 1. Calculate correlations with bootstrapping for TOTAL (AI vs Human)
    print("\nCalculating correlations with bootstrapping for AI (Total)...")
    ai_bootstrap = bootstrap_correlation(
        df_ai_processed, 
        available_features, 
        "accuracy_score", 
        sample_size=n_samples,
        n_iterations=n_bootstrap,
        conf_level=confidence
    )
    
    # Save results
    point_correlations["AI_Total"] = [ai_bootstrap[feature]['correlation'] for feature in available_features]
    lower_ci["AI_Total"] = [ai_bootstrap[feature]['lower_ci'] for feature in available_features]
    upper_ci["AI_Total"] = [ai_bootstrap[feature]['upper_ci'] for feature in available_features]
    
    print("Calculating correlations with bootstrapping for Human (Total)...")
    human_bootstrap = bootstrap_correlation(
        df_human_processed, 
        available_features, 
        "accuracy_score", 
        sample_size=n_samples,
        n_iterations=n_bootstrap,
        conf_level=confidence
    )
    
    # Save results
    point_correlations["Human_Total"] = [human_bootstrap[feature]['correlation'] for feature in available_features]
    lower_ci["Human_Total"] = [human_bootstrap[feature]['lower_ci'] for feature in available_features]
    upper_ci["Human_Total"] = [human_bootstrap[feature]['upper_ci'] for feature in available_features]
    
    # 2. Now calculate correlations for each LLM type
    for llm_value in [0, 1, 2]:
        llm_name = llm_types[llm_value]
        print(f"\nProcessing correlations for {llm_name} (LLM={llm_value})...")
        
        # Filter AI dataframe by LLM type
        if llm_column in df_ai_processed.columns:
            df_ai_filtered = df_ai_processed[df_ai_processed[llm_column] == llm_value].copy()
            if len(df_ai_filtered) > 10:  # Ensure enough data for bootstrap
                print(f"  - AI: {len(df_ai_filtered)} evaluations found")
                
                # Calculate correlations with bootstrap
                ai_llm_bootstrap = bootstrap_correlation(
                    df_ai_filtered, 
                    available_features, 
                    "accuracy_score", 
                    sample_size=n_samples,
                    n_iterations=n_bootstrap,
                    conf_level=confidence
                )
                
                # Save results - FIXED: Corrected upper_ci assignment
                point_correlations[f"AI_LLM{llm_value}"] = [ai_llm_bootstrap[feature]['correlation'] for feature in available_features]
                lower_ci[f"AI_LLM{llm_value}"] = [ai_llm_bootstrap[feature]['lower_ci'] for feature in available_features]
                upper_ci[f"AI_LLM{llm_value}"] = [ai_llm_bootstrap[feature]['upper_ci'] for feature in available_features]
            else:
                print(f"  - AI: Not enough evaluations for LLM={llm_value} (minimum 10)")
                point_correlations[f"AI_LLM{llm_value}"] = np.nan
                lower_ci[f"AI_LLM{llm_value}"] = np.nan
                upper_ci[f"AI_LLM{llm_value}"] = np.nan
        
        # Filter Human dataframe by LLM type
        if llm_column in df_human_processed.columns:
            df_human_filtered = df_human_processed[df_human_processed[llm_column] == llm_value].copy()
            if len(df_human_filtered) > 10:  # Ensure enough data for bootstrap
                print(f"  - Human: {len(df_human_filtered)} evaluations found")
                
                # Calculate correlations with bootstrap
                human_llm_bootstrap = bootstrap_correlation(
                    df_human_filtered, 
                    available_features, 
                    "accuracy_score", 
                    sample_size=n_samples,
                    n_iterations=n_bootstrap,
                    conf_level=confidence
                )
                
                # Save results
                point_correlations[f"Human_LLM{llm_value}"] = [human_llm_bootstrap[feature]['correlation'] for feature in available_features]
                lower_ci[f"Human_LLM{llm_value}"] = [human_llm_bootstrap[feature]['lower_ci'] for feature in available_features]
                upper_ci[f"Human_LLM{llm_value}"] = [human_llm_bootstrap[feature]['upper_ci'] for feature in available_features]
            else:
                print(f"  - Human: Not enough evaluations for LLM={llm_value} (minimum 10)")
                point_correlations[f"Human_LLM{llm_value}"] = np.nan
                lower_ci[f"Human_LLM{llm_value}"] = np.nan
                upper_ci[f"Human_LLM{llm_value}"] = np.nan
    
    # Sort features by category for visualization
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        ordered_features.extend([f for f in feature_group if f in available_features])
    
    # Reorder dataframe rows
    sorted_corr = point_correlations.loc[ordered_features, :]
    sorted_lower = lower_ci.loc[ordered_features, :]
    sorted_upper = upper_ci.loc[ordered_features, :]
    
    # Create a DataFrame with complete results - FIXED: Using concat properly with pandas 
    result_df = pd.DataFrame()
    for col in sorted_corr.columns:
        for feature in sorted_corr.index:
            corr = sorted_corr.loc[feature, col]
            lower = sorted_lower.loc[feature, col]
            upper = sorted_upper.loc[feature, col]
            
            # Determine if correlation is significant (CI does not include zero)
            significant = False
            if not (np.isnan(lower) or np.isnan(upper)):
                significant = (lower > 0 and upper > 0) or (lower < 0 and upper < 0)
                
            new_row = pd.DataFrame({
                'Feature': [feature],
                'Group': [col],
                'Correlation': [corr],
                'Lower_CI': [lower],
                'Upper_CI': [upper],
                'Significant': [significant]
            })
            result_df = pd.concat([result_df, new_row], ignore_index=True)
    
    # Create Plotly heatmap
    fig = create_plotly_correlation_heatmap(
        sorted_corr, 
        sorted_lower, 
        sorted_upper, 
        confidence=confidence
    )
    
    # Save the image and HTML
    fig.write_image('bootstrap_correlation_heatmap.png', scale=2)
    fig.write_html('bootstrap_correlation_heatmap.html')
    
    # Save detailed results
    result_df.to_csv('bootstrap_correlation_results.csv', index=False)
    
    return fig, result_df
def create_plotly_correlation_heatmap(point_correlations, lower_ci, upper_ci, confidence=0.95):
    """
    Creates a publication-quality correlation heatmap with confidence intervals using Plotly.
    Nature style with clustering.
    
    Args:
        point_correlations: DataFrame with point correlations
        lower_ci: DataFrame with lower bounds of confidence intervals
        upper_ci: DataFrame with upper bounds of confidence intervals
        confidence: Confidence level for intervals (default 0.95)
        
    Returns:
        Plotly figure object
    """
    import plotly.graph_objects as go
    import numpy as np
    import pandas as pd
    from scipy.cluster import hierarchy
    from scipy.spatial import distance
    
    # Rename columns for better clarity
    column_mapping = {
        'AI_Total': 'AI (All)',
        'Human_Total': 'Human (All)',
        'AI_LLM0': 'AI (Human)',
        'Human_LLM0': 'Human (Human)',
        'AI_LLM1': 'AI (AI)',
        'Human_LLM1': 'Human (AI)',
        'AI_LLM2': 'AI (CoT)',
        'Human_LLM2': 'Human (CoT)'
    }
    
    # Create copies to avoid modifying the originals
    sorted_corr = point_correlations.copy()
    sorted_lower = lower_ci.copy()
    sorted_upper = upper_ci.copy()
    
    # Rename columns
    sorted_corr = sorted_corr.rename(columns=column_mapping)
    sorted_lower = sorted_lower.rename(columns=column_mapping)
    sorted_upper = sorted_upper.rename(columns=column_mapping)
    
    # Define feature categories
    count_features = ['word_count', 'sentence_count']
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Format feature names for better readability
    feature_name_mapping = {
        'word_count': 'Word Count',
        'sentence_count': 'Sentence Count',
        'avg_word_length': 'Word Length',
        'avg_sentence_length': 'Sentence Length',
        'reading_ease': 'Readability',
        'grade_level': 'Grade Level',
        'connector_count': 'Connectors',
        'unique_words_ratio': 'Unique Words',
        'lexical_diversity': 'Lexical Diversity',
        'verb_ratio': 'Verbs',
        'noun_ratio': 'Nouns',
        'adj_ratio': 'Adjectives',
        'adv_ratio': 'Adverbs',
        'pronoun_ratio': 'Pronouns',
        'named_entity_count': 'Named Entities',
        'dependency_distance': 'Dependency Dist.'
    }
    
    # Apply the mapping to create a new index
    sorted_corr.index = [feature_name_mapping.get(feature, feature) for feature in sorted_corr.index]
    sorted_lower.index = [feature_name_mapping.get(feature, feature) for feature in sorted_lower.index]
    sorted_upper.index = [feature_name_mapping.get(feature, feature) for feature in sorted_upper.index]
    
    # Get available features in order (ensuring we use the formatted names)
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        for feature in feature_group:
            formatted_name = feature_name_mapping.get(feature, feature)
            if formatted_name in sorted_corr.index:
                ordered_features.append(formatted_name)
    
    # Reorder dataframe rows
    sorted_corr = sorted_corr.loc[ordered_features, :]
    sorted_lower = sorted_lower.loc[ordered_features, :]
    sorted_upper = sorted_upper.loc[ordered_features, :]
    
    # === CLUSTERING DE COLUMNAS ===
    print("\n===== COLUMN CLUSTERING =====")
    
    # Convert to numeric and fill NaNs for clustering
    numeric_corr = sorted_corr.astype(float).fillna(0)
    
    # Transpose for column clustering
    col_matrix = numeric_corr.T.values
    
    # Compute linkage matrix for columns
    col_linkage = hierarchy.linkage(col_matrix, method='ward', metric='euclidean')
    
    # Compute dendrogram
    col_dendrogram = hierarchy.dendrogram(col_linkage, no_plot=True)
    
    # Get the order of columns after clustering
    col_order = col_dendrogram['leaves']
    clustered_cols = [sorted_corr.columns[i] for i in col_order]
    
    print(f"Column order after clustering: {', '.join(clustered_cols)}")
    
    # Reorder columns based on clustering
    sorted_corr = sorted_corr[clustered_cols]
    sorted_lower = sorted_lower[clustered_cols]
    sorted_upper = sorted_upper[clustered_cols]
    
    # Determine significance (CI does not include zero)
    significant = ((sorted_lower > 0) & (sorted_upper > 0)) | ((sorted_lower < 0) & (sorted_upper < 0))
    
    # Define category boundaries in the formatted index
    count_formatted = [feature_name_mapping.get(f, f) for f in count_features 
                      if feature_name_mapping.get(f, f) in sorted_corr.index]
    surface_formatted = [feature_name_mapping.get(f, f) for f in surface_features 
                        if feature_name_mapping.get(f, f) in sorted_corr.index]
    
    count_end = len(count_formatted)
    surface_end = count_end + len(surface_formatted)
    
    # Prepare hover text
    hover_texts = []
    for i, feature in enumerate(sorted_corr.index):
        feature_texts = []
        for j, column in enumerate(sorted_corr.columns):
            corr_value = sorted_corr.loc[feature, column]
            lower = sorted_lower.loc[feature, column]
            upper = sorted_upper.loc[feature, column]
            is_sig = significant.loc[feature, column]
            
            if not np.isnan(corr_value):
                text = f"Feature: {feature}<br>"
                text += f"Group: {column}<br>"
                text += f"Correlation: {corr_value:.3f}<br>"
                text += f"CI ({confidence*100:.0f}%): [{lower:.3f}, {upper:.3f}]<br>"
                text += f"Statistically significant: {'Yes' if is_sig else 'No'}"
                feature_texts.append(text)
            else:
                feature_texts.append("")
        hover_texts.append(feature_texts)
    
    # Create base figure
    fig = go.Figure()
    
    # Add heatmap
    fig.add_trace(go.Heatmap(
        z=sorted_corr.values,
        x=sorted_corr.columns,
        y=sorted_corr.index,
        colorscale='RdBu_r',
        zmid=0,
        zmin=-1,
        zmax=1,
        hoverinfo='text',
        hovertext=hover_texts,
        showscale=True,
        colorbar=dict(
            title="Correlation",
            titlefont=dict(size=22, family="Arial"),
            tickfont=dict(size=20, family="Arial"),
            len=0.7,
            thickness=20
        )
    ))
    
    # Add annotations for correlation values - ESTILO NATURE
    annotations = []
    for i, feature in enumerate(sorted_corr.index):
        for j, column in enumerate(sorted_corr.columns):
            corr_value = sorted_corr.loc[feature, column]
            is_sig = significant.loc[feature, column]
            
            if not np.isnan(corr_value):
                text = f"{corr_value:.2f}"
                if is_sig:
                    text += "*"
                
                # Text color based on value for better readability
                text_color = 'white' if abs(corr_value) > 0.5 else 'black'
                
                annotations.append(dict(
                    x=j,
                    y=i,
                    text=text,
                    font=dict(
                        color=text_color, 
                        size=20,  # MÁS GRANDE
                        family="Arial, sans-serif"
                    ),
                    showarrow=False,
                    xref='x',
                    yref='y'
                ))
    
    # Add horizontal lines to separate categories - MÁS GRUESAS
    shapes = [
        dict(
            type="line",
            x0=-0.5,
            y0=count_end - 0.5,
            x1=len(sorted_corr.columns) - 0.5,
            y1=count_end - 0.5,
            line=dict(color="black", width=2)
        ),
        dict(
            type="line",
            x0=-0.5,
            y0=surface_end - 0.5,
            x1=len(sorted_corr.columns) - 0.5,
            y1=surface_end - 0.5,
            line=dict(color="black", width=2)
        )
    ]
    
    # Add note about statistical significance
    annotations.append(dict(
        x=-0.15,
        y=-0.25,
        xref="paper",
        yref="paper",
        text="* Statistically significant (CI excludes zero)",
        showarrow=False,
        font=dict(size=18, family="Arial"),
        align="left",
        xanchor="left"
    ))
    
    # Update layout - ESTILO NATURE
    fig.update_layout(
        title=dict(
            text=f"Feature Correlations with Accuracy Score (Clustered)",
            font=dict(size=28, family="Arial"),
            x=0.5,
            xanchor="center",
            y=0.98
        ),
        height=800,  # Más grande
        width=1000,  # Más grande
        xaxis=dict(
            title='',
            tickfont=dict(size=20, family="Arial"),  # MÁS GRANDE
            tickangle=-45,
            showgrid=False,  # SIN CUADRÍCULA
            showline=True,
            linewidth=2,
            linecolor='black',
            mirror=False,
            zeroline=False
        ),
        yaxis=dict(
            title='',
            tickfont=dict(size=20, family="Arial"),  # MÁS GRANDE
            autorange="reversed",
            showgrid=False,  # SIN CUADRÍCULA
            showline=True,
            linewidth=2,
            linecolor='black',
            mirror=False,
            zeroline=False
        ),
        template="plotly_white",
        shapes=shapes,
        annotations=annotations,
        margin=dict(l=150, r=120, t=80, b=120),
        paper_bgcolor='white',
        plot_bgcolor='white'
    )
    
    return fig
def run_bootstrap_correlation_analysis(df_ai, df_human, text_column='response', llm_column='LLM', n_bootstrap=1000, n_samples=50, confidence=0.95):
    """
    Runs comparative analysis between AI and Human with bootstrapping.
    
    Args:
        df_ai: DataFrame with AI evaluations
        df_human: DataFrame with human evaluations
        text_column: Name of the column containing the text to analyze
        llm_column: Name of the column identifying the LLM type
        n_bootstrap: Number of bootstrap samples
        confidence: Confidence level (0-1)
        
    Returns:
        Tuple with (figure, results dataframe)
    """
    print(f"Starting comparative analysis with bootstrapping ({n_bootstrap} iterations with {n_samples} samples, {confidence*100:.0f}% confidence)...")
    
    # Run the comparative analysis
    fig, results_df = create_comparative_correlation_heatmap_with_bootstrap(
        df_ai, df_human, text_column, llm_column, n_bootstrap, n_samples, confidence
    )
    
    # Print a summary of findings
    print("\n===== BOOTSTRAP ANALYSIS SUMMARY =====\n")
    
    # Print available groups
    groups = results_df['Group'].unique()
    print(f"Analyzed groups: {', '.join(groups)}")
    
    # Define feature categories
    count_features = ['word_count', 'sentence_count']
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Find significant correlations
    significant_results = results_df[results_df['Significant'] == True]
    print(f"\nFound {len(significant_results)} statistically significant correlations")
    
    # Summary by group
    for group in groups:
        group_results = results_df[results_df['Group'] == group]
        sig_count = sum(group_results['Significant'])
        print(f"\n{group}: {sig_count} significant correlations out of {len(group_results)}")
        
        # Top positive significant correlations
        pos_sig = group_results[(group_results['Significant']) & (group_results['Correlation'] > 0)]
        if len(pos_sig) > 0:
            top_pos = pos_sig.sort_values('Correlation', ascending=False).head(3)
            print("  Top positive significant correlations:")
            for _, row in top_pos.iterrows():
                print(f"    - {row['Feature']}: {row['Correlation']:.3f} (CI: {row['Lower_CI']:.3f} to {row['Upper_CI']:.3f})")
                
        # Top negative significant correlations
        neg_sig = group_results[(group_results['Significant']) & (group_results['Correlation'] < 0)]
        if len(neg_sig) > 0:
            top_neg = neg_sig.sort_values('Correlation', ascending=True).head(3)
            print("  Top negative significant correlations:")
            for _, row in top_neg.iterrows():
                print(f"    - {row['Feature']}: {row['Correlation']:.3f} (CI: {row['Lower_CI']:.3f} to {row['Upper_CI']:.3f})")
    
    # Analysis by feature category
    print("\n===== ANALYSIS BY FEATURE CATEGORY =====")
    
    for group in groups:
        print(f"\n{group}:")
        
        # Count significant by category
        for category_name, category_features in [
            ("COUNT", count_features), 
            ("SURFACE", surface_features), 
            ("STRUCTURE", structure_features)
        ]:
            category_results = results_df[
                (results_df['Group'] == group) & 
                (results_df['Feature'].isin(category_features))
            ]
            
            if len(category_results) > 0:
                sig_count = sum(category_results['Significant'])
                sig_percent = sig_count / len(category_results) * 100
                
                # Calculate average correlation (absolute)
                abs_corr = category_results['Correlation'].abs().mean()
                
                print(f"  {category_name}: {sig_count}/{len(category_results)} significant ({sig_percent:.1f}%), |r| average = {abs_corr:.3f}")
    
    # Save detailed results
    results_df.to_csv('bootstrap_correlation_results.csv', index=False)
    print("\nDetailed results saved to 'bootstrap_correlation_results.csv'")
    
    return fig, results_df

In [ ]:
# Para ejecutar el análisis comparativo entre AI y Human
fig, corr_df = create_comparative_correlation_heatmap_with_bootstrap(
    df_ai=df_ai,  # Tu dataframe de evaluaciones de IA
    df_human=df_human,  # Tu dataframe de evaluaciones de humanos
    text_column='response',  # Columna con el texto a analizar
    llm_column='LLM'  # Columna que identifica el tipo de LLM
)
fig.show()

In [ ]:
def create_simplified_correlation_heatmap_with_bootstrap(df_ai, df_human, text_column='response', n_bootstrap=1000, n_samples=50, confidence=0.95):
    """
    Creates a simplified correlation heatmap with confidence intervals (only AI vs Human totals).
    Paper-ready visualization with clean academic styling.
    
    Args:
        df_ai: DataFrame with AI evaluations
        df_human: DataFrame with human evaluations
        text_column: Name of the column containing the text to analyze
        n_bootstrap: Number of bootstrap samples for confidence intervals
        n_samples: Sample size for each bootstrap iteration
        confidence: Confidence level for intervals (default 0.95)
        
    Returns:
        Plotly figure with comparative heatmap and DataFrame with correlations and intervals
    """
    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go
    from scipy.stats import spearmanr
    from tqdm import tqdm
    
    def bootstrap_correlation(data, features, target, sample_size=50, n_iterations=1000, conf_level=0.95):
        """
        Calculates correlations using bootstrap resampling technique.
        """
        import numpy as np
        import pandas as pd
        from scipy.stats import spearmanr
        
        # Verify that sample_size is not larger than available data
        n = len(data)
        if sample_size > n:
            print(f"WARNING: Sample size ({sample_size}) larger than available data ({n}). Adjusting to {n}.")
            sample_size = n
        
        correlations = []
        
        # Perform n_iterations of subsampling
        for _ in range(n_iterations):
            # Select sample_size random samples (with replacement)
            indices = np.random.choice(range(n), size=sample_size, replace=True)
            bootstrap_sample = data.iloc[indices]
            
            # Calculate correlations for this sample
            corr_sample = {}
            for feature in features:
                # Verify if there's enough valid data
                valid_data = ~(bootstrap_sample[feature].isna() | bootstrap_sample[target].isna())
                if valid_data.sum() < 3:  # Need at least 3 points for correlation
                    corr_sample[feature] = np.nan
                    continue
                    
                # Use scipy.stats.spearmanr to get only the coefficient
                corr, _ = spearmanr(bootstrap_sample.loc[valid_data, feature], 
                                bootstrap_sample.loc[valid_data, target])
                corr_sample[feature] = corr
            
            correlations.append(corr_sample)
        
        # Convert to DataFrame for easier calculation
        corr_df = pd.DataFrame(correlations)
        
        # Calculate point correlation and confidence intervals
        alpha = 1 - conf_level
        result = {}
        for feature in features:
            # Filter NaN values
            feature_corrs = corr_df[feature].dropna()
            
            if len(feature_corrs) == 0:
                result[feature] = {
                    'correlation': np.nan,
                    'lower_ci': np.nan,
                    'upper_ci': np.nan,
                    'sample_count': 0
                }
                continue
            
            # Point estimate (bootstrap average)
            point_estimate = feature_corrs.mean()
            
            # Confidence intervals
            if len(feature_corrs) >= 3:
                lower_bound = np.percentile(feature_corrs, alpha/2 * 100)
                upper_bound = np.percentile(feature_corrs, (1 - alpha/2) * 100)
            else:
                lower_bound = np.nan
                upper_bound = np.nan
            
            result[feature] = {
                'correlation': point_estimate,
                'lower_ci': lower_bound,
                'upper_ci': upper_bound,
                'sample_count': len(feature_corrs)
            }
        
        return result
    
    # Prepare the dataframes
    print("Processing AI dataframe...")
    df_ai_processed = prepare_dataframe_for_analysis(df_ai, text_column=text_column)
    
    print("Processing Human dataframe...")
    df_human_processed = prepare_dataframe_for_analysis(df_human, text_column=text_column)
    
    # Define feature categories
    count_features = ['word_count', 'sentence_count']
    
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # All features
    all_features = count_features + surface_features + structure_features
    
    # Check which features are available in both dataframes
    available_features = [f for f in all_features 
                         if f in df_ai_processed.columns and f in df_human_processed.columns]
    
    # Check if accuracy_score is available in both dataframes
    if "accuracy_score" not in df_ai_processed.columns:
        raise ValueError("'accuracy_score' not found in AI dataframe")
    if "accuracy_score" not in df_human_processed.columns:
        raise ValueError("'accuracy_score' not found in Human dataframe")
    
    # Create dataframes to store correlations and intervals
    point_correlations = pd.DataFrame(index=available_features)
    lower_ci = pd.DataFrame(index=available_features)
    upper_ci = pd.DataFrame(index=available_features)
    
    # Calculate correlations with bootstrapping for AI (Total)
    print("\nCalculating correlations with bootstrapping for AI evaluators...")
    ai_bootstrap = bootstrap_correlation(
        df_ai_processed, 
        available_features, 
        "accuracy_score", 
        sample_size=n_samples,
        n_iterations=n_bootstrap,
        conf_level=confidence
    )
    
    # Save AI results
    point_correlations["AI"] = [ai_bootstrap[feature]['correlation'] for feature in available_features]
    lower_ci["AI"] = [ai_bootstrap[feature]['lower_ci'] for feature in available_features]
    upper_ci["AI"] = [ai_bootstrap[feature]['upper_ci'] for feature in available_features]
    
    # Calculate correlations with bootstrapping for Human (Total)
    print("Calculating correlations with bootstrapping for Human evaluators...")
    human_bootstrap = bootstrap_correlation(
        df_human_processed, 
        available_features, 
        "accuracy_score", 
        sample_size=n_samples,
        n_iterations=n_bootstrap,
        conf_level=confidence
    )
    
    # Save Human results
    point_correlations["Human"] = [human_bootstrap[feature]['correlation'] for feature in available_features]
    lower_ci["Human"] = [human_bootstrap[feature]['lower_ci'] for feature in available_features]
    upper_ci["Human"] = [human_bootstrap[feature]['upper_ci'] for feature in available_features]
    
    # Sort features by category for visualization
    ordered_features = []
    for feature_group in [count_features, surface_features, structure_features]:
        ordered_features.extend([f for f in feature_group if f in available_features])
    
    # Reorder dataframe rows
    sorted_corr = point_correlations.loc[ordered_features, :]
    sorted_lower = lower_ci.loc[ordered_features, :]
    sorted_upper = upper_ci.loc[ordered_features, :]
    
    # Create a DataFrame with complete results
    result_df = pd.DataFrame()
    for col in sorted_corr.columns:
        for feature in sorted_corr.index:
            corr = sorted_corr.loc[feature, col]
            lower = sorted_lower.loc[feature, col]
            upper = sorted_upper.loc[feature, col]
            
            # Determine if correlation is significant (CI does not include zero)
            significant = False
            if not (np.isnan(lower) or np.isnan(upper)):
                significant = (lower > 0 and upper > 0) or (lower < 0 and upper < 0)
                
            new_row = pd.DataFrame({
                'Feature': [feature],
                'Evaluator': [col],
                'Correlation': [corr],
                'Lower_CI': [lower],
                'Upper_CI': [upper],
                'Significant': [significant]
            })
            result_df = pd.concat([result_df, new_row], ignore_index=True)
    
    # Create paper-ready Plotly heatmap
    fig = create_paper_ready_heatmap(
        sorted_corr, 
        sorted_lower, 
        sorted_upper, 
        confidence=confidence
    )
    
    return fig, result_df


def create_paper_ready_heatmap(point_correlations, lower_ci, upper_ci, confidence=0.95):
    """
    Creates a clean publication-ready correlation heatmap optimized for academic papers.
    
    Args:
        point_correlations: DataFrame with point correlations
        lower_ci: DataFrame with lower bounds of confidence intervals
        upper_ci: DataFrame with upper bounds of confidence intervals
        confidence: Confidence level for intervals (default 0.95)
        
    Returns:
        Plotly figure object optimized for academic publications
    """
    import plotly.graph_objects as go
    import numpy as np
    import pandas as pd
    
    # Define feature categories for better organization
    count_features = ['word_count', 'sentence_count']
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    
    # Format feature names for academic presentation
    feature_name_mapping = {
        'word_count': 'Word Count',
        'sentence_count': 'Sentence Count',
        'avg_word_length': 'Average Word Length',
        'avg_sentence_length': 'Average Sentence Length',
        'reading_ease': 'Reading Ease',
        'grade_level': 'Grade Level',
        'connector_count': 'Connector Count',
        'unique_words_ratio': 'Unique Words Ratio',
        'lexical_diversity': 'Lexical Diversity',
        'verb_ratio': 'Verb Ratio',
        'noun_ratio': 'Noun Ratio',
        'adj_ratio': 'Adjective Ratio',
        'adv_ratio': 'Adverb Ratio',
        'pronoun_ratio': 'Pronoun Ratio',
        'named_entity_count': 'Named Entity Count',
        'dependency_distance': 'Dependency Distance'
    }
    
    # Apply the mapping to create formatted labels
    sorted_corr = point_correlations.copy()
    sorted_lower = lower_ci.copy()
    sorted_upper = upper_ci.copy()
    
    sorted_corr.index = [feature_name_mapping.get(feature, feature) for feature in sorted_corr.index]
    sorted_lower.index = [feature_name_mapping.get(feature, feature) for feature in sorted_lower.index]
    sorted_upper.index = [feature_name_mapping.get(feature, feature) for feature in sorted_upper.index]
    
    # Determine significance (CI does not include zero)
    significant = ((sorted_lower > 0) & (sorted_upper > 0)) | ((sorted_lower < 0) & (sorted_upper < 0))
    
    # Prepare hover text with detailed information
    hover_texts = []
    for i, feature in enumerate(sorted_corr.index):
        feature_texts = []
        for j, column in enumerate(sorted_corr.columns):
            corr_value = sorted_corr.loc[feature, column]
            lower = sorted_lower.loc[feature, column]
            upper = sorted_upper.loc[feature, column]
            is_sig = significant.loc[feature, column]
            
            if not np.isnan(corr_value):
                text = f"<b>{feature}</b><br>"
                text += f"<b>Evaluator:</b> {column}<br>"
                text += f"<b>Correlation:</b> {corr_value:.3f}<br>"
                text += f"<b>{confidence*100:.0f}% CI:</b> [{lower:.3f}, {upper:.3f}]<br>"
                text += f"<b>Significant:</b> {'Yes' if is_sig else 'No'}"
                feature_texts.append(text)
            else:
                feature_texts.append("")
        hover_texts.append(feature_texts)
    
    # Create base figure
    fig = go.Figure()
    
    # Add heatmap without colorbar (cleaner for papers)
    fig.add_trace(go.Heatmap(
        z=sorted_corr.values,
        x=sorted_corr.columns,
        y=sorted_corr.index,
        colorscale='RdBu_r',
        zmid=0,
        zmin=-0.8,
        zmax=0.8,
        hoverinfo='text',
        hovertext=hover_texts,
        showscale=False,  # No colorbar
        colorbar=None     # Ensure no colorbar
    ))
    
    # Add annotations for correlation values with much larger fonts
    annotations = []
    for i, feature in enumerate(sorted_corr.index):
        for j, column in enumerate(sorted_corr.columns):
            corr_value = sorted_corr.loc[feature, column]
            is_sig = significant.loc[feature, column]
            
            if not np.isnan(corr_value):
                # Format correlation value
                text = f"{corr_value:.3f}"
                if is_sig:
                    text = f"<b>{text}*</b>"
                
                # Text color for readability
                text_color = 'white' if abs(corr_value) > 0.4 else 'black'
                
                annotations.append(dict(
                    x=j,
                    y=i,
                    text=text,
                    font=dict(
                        color=text_color, 
                        size=28,  # Much larger font size for correlation values
                        family="Arial Black"
                    ),
                    showarrow=False,
                    xref='x',
                    yref='y'
                ))
    
    # Calculate category boundaries for visual separation
    count_formatted = [feature_name_mapping.get(f, f) for f in count_features 
                      if feature_name_mapping.get(f, f) in sorted_corr.index]
    surface_formatted = [feature_name_mapping.get(f, f) for f in surface_features 
                        if feature_name_mapping.get(f, f) in sorted_corr.index]
    
    count_end = len(count_formatted)
    surface_end = count_end + len(surface_formatted)
    
    # Add category separation lines (thicker and more prominent)
    shapes = []
    if count_end > 0 and count_end < len(sorted_corr.index):
        shapes.append(dict(
            type="line",
            x0=-0.5,
            y0=count_end - 0.5,
            x1=len(sorted_corr.columns) - 0.5,
            y1=count_end - 0.5,
            line=dict(color="black", width=2)
        ))
    
    if surface_end > count_end and surface_end < len(sorted_corr.index):
        shapes.append(dict(
            type="line",
            x0=-0.5,
            y0=surface_end - 0.5,
            x1=len(sorted_corr.columns) - 0.5,
            y1=surface_end - 0.5,
            line=dict(color="black", width=2)
        ))
    
    # Statistical significance note (larger font and positioned better)
    sig_note = dict(
        x=0.5,
        y=-0.03,
        xref="paper",
        yref="paper",
        text=f"* <i>p</i> < {1-confidence:.2f}",
        showarrow=False,
        font=dict(size=20, family="Arial", weight="bold")  # Much larger font
      
    )
    
    # Only use correlation value annotations + significance note
    all_annotations = annotations 
    
    # Update layout with much larger fonts and better spacing
    fig.update_layout(
        height=1000,  # Increased height for better spacing
        width=600,    # Keep width reasonable
        xaxis=dict(
            title=dict(
                text="<b>Evaluator</b>",
                font=dict(size=26, family="Arial", weight="bold")  # Much larger title
            ),
            tickfont=dict(size=24, family="Arial", weight="bold"),  # Much larger tick labels
            side="bottom"
        ),
        yaxis=dict(
            title="",
            tickfont=dict(size=20, family="Arial", weight="bold"),  # Much larger feature names
            autorange="reversed"
        ),
        template="plotly_white",
        shapes=shapes,
        annotations=all_annotations,
        margin=dict(l=250, r=50, t=50, b=100),  # Much larger left margin so labels don't overlap plot
        paper_bgcolor='white',
        plot_bgcolor='white',
        font=dict(family="Arial", size=20),
        showlegend=False  # Explicitly no legend
    )
    
    return fig


def run_simplified_bootstrap_analysis(df_ai, df_human, text_column='response', n_bootstrap=1000, n_samples=50, confidence=0.95):
    """
    Runs simplified comparative analysis between AI and Human evaluators with bootstrapping.
    
    Args:
        df_ai: DataFrame with AI evaluations
        df_human: DataFrame with human evaluations
        text_column: Name of the column containing the text to analyze
        n_bootstrap: Number of bootstrap samples
        n_samples: Sample size for each bootstrap iteration
        confidence: Confidence level (0-1)
        
    Returns:
        Tuple with (figure, results dataframe)
    """
    print(f"Starting simplified comparative analysis...")
    print(f"Bootstrap parameters: {n_bootstrap} iterations, {n_samples} samples per iteration, {confidence*100:.0f}% confidence")
    
    # Run the comparative analysis
    fig, results_df = create_simplified_correlation_heatmap_with_bootstrap(
        df_ai, df_human, text_column, n_bootstrap, n_samples, confidence
    )
    
    # Print summary of findings
    print("\n===== SIMPLIFIED BOOTSTRAP ANALYSIS SUMMARY =====\n")
    
    # Count significant correlations by evaluator
    for evaluator in ['AI', 'Human']:
        evaluator_results = results_df[results_df['Evaluator'] == evaluator]
        sig_count = sum(evaluator_results['Significant'])
        total_features = len(evaluator_results)
        sig_percentage = (sig_count / total_features) * 100
        
        print(f"{evaluator} Evaluators:")
        print(f"  - Significant correlations: {sig_count}/{total_features} ({sig_percentage:.1f}%)")
        
        # Top positive correlations
        pos_corr = evaluator_results[evaluator_results['Correlation'] > 0].sort_values('Correlation', ascending=False).head(3)
        if len(pos_corr) > 0:
            print("  - Top positive correlations:")
            for _, row in pos_corr.iterrows():
                sig_marker = "*" if row['Significant'] else ""
                print(f"    • {row['Feature']}: r = {row['Correlation']:.3f} [{row['Lower_CI']:.3f}, {row['Upper_CI']:.3f}]{sig_marker}")
        
        # Top negative correlations
        neg_corr = evaluator_results[evaluator_results['Correlation'] < 0].sort_values('Correlation', ascending=True).head(3)
        if len(neg_corr) > 0:
            print("  - Top negative correlations:")
            for _, row in neg_corr.iterrows():
                sig_marker = "*" if row['Significant'] else ""
                print(f"    • {row['Feature']}: r = {row['Correlation']:.3f} [{row['Lower_CI']:.3f}, {row['Upper_CI']:.3f}]{sig_marker}")
        print()
    
    # Compare correlations between AI and Human
    print("===== AI vs HUMAN COMPARISON =====")
    ai_results = results_df[results_df['Evaluator'] == 'AI'].set_index('Feature')
    human_results = results_df[results_df['Evaluator'] == 'Human'].set_index('Feature')
    
    comparison_df = pd.DataFrame({
        'Feature': ai_results.index,
        'AI_Correlation': ai_results['Correlation'].values,
        'Human_Correlation': human_results['Correlation'].values,
        'AI_Significant': ai_results['Significant'].values,
        'Human_Significant': human_results['Significant'].values
    })
    
    comparison_df['Difference'] = comparison_df['AI_Correlation'] - comparison_df['Human_Correlation']
    comparison_df['Both_Significant'] = comparison_df['AI_Significant'] & comparison_df['Human_Significant']
    
    # Features with largest differences
    print("Features with largest correlation differences (AI - Human):")
    top_diff = comparison_df.sort_values('Difference', key=abs, ascending=False).head(5)
    for _, row in top_diff.iterrows():
        print(f"  • {row['Feature']}: Δr = {row['Difference']:+.3f} (AI: {row['AI_Correlation']:.3f}, Human: {row['Human_Correlation']:.3f})")
    
    # Save results
    results_df.to_csv('simplified_bootstrap_correlation_results.csv', index=False)
    comparison_df.to_csv('ai_human_correlation_comparison.csv', index=False)
    
    # Save figure with updated dimensions
    fig.write_image('simplified_correlation_heatmap.png', scale=2, width=600, height=1000)
    fig.write_html('simplified_correlation_heatmap.html')
    
    print(f"\nResults saved:")
    print("- simplified_bootstrap_correlation_results.csv")
    print("- ai_human_correlation_comparison.csv") 
    print("- simplified_correlation_heatmap.png")
    print("- simplified_correlation_heatmap.html")
    
    return fig, results_df

In [ ]:
# Para ejecutar el análisis comparativo entre AI y Human
fig, corr_df = create_simplified_correlation_heatmap_with_bootstrap(
    df_ai=df_ai,  # Tu dataframe de evaluaciones de IA
    df_human=df_human,  # Tu dataframe de evaluaciones de humanos
    text_column='response',  # Columna con el texto a analizar
    
)
fig.show()

In [ ]:
def create_all_evaluators_linguistic_heatmap_with_bootstrap(
    eval_collections,
    eval_names,
    score_column='accuracy_score',
    n_bootstrap=1000,
    n_samples=50,
    confidence=0.95,
    idk=False
):
    """
    Crea un heatmap de correlaciones entre características lingüísticas de las RESPUESTAS
    y las PUNTUACIONES que da cada evaluador.
    """
    from scipy.stats import spearmanr
    
    def format_label(label):
        """Format labels: remove underscores, capitalize properly"""
        # Replace underscores with spaces
        formatted = label.replace('_', ' ')
        # Capitalize each word
        formatted = ' '.join(word.capitalize() for word in formatted.split())
        return formatted
    
    def bootstrap_correlation(df, feature, score_col, n_bootstrap, n_samples):
        """Calcula correlación con bootstrap."""
        correlations = []
        
        df_clean = df[[feature, score_col]].dropna()
        
        if len(df_clean) < n_samples:
            return {'correlation': np.nan, 'lower_ci': np.nan, 'upper_ci': np.nan}
        
        for _ in range(n_bootstrap):
            sample = df_clean.sample(n=n_samples, replace=True)
            try:
                corr, _ = spearmanr(sample[feature], sample[score_col])
                if not np.isnan(corr):
                    correlations.append(corr)
            except:
                continue
        
        if len(correlations) == 0:
            return {'correlation': np.nan, 'lower_ci': np.nan, 'upper_ci': np.nan}
        
        point_estimate = np.mean(correlations)
        alpha = 1 - confidence
        lower_bound = np.percentile(correlations, (alpha/2) * 100)
        upper_bound = np.percentile(correlations, (1 - alpha/2) * 100)
        
        return {
            'correlation': point_estimate,
            'lower_ci': lower_bound,
            'upper_ci': upper_bound
        }
    
    def combine_answer_and_bibliography(row):
        if pd.notna(row['bibliography_sources']):
            return row['answer'] + '\n\nReferences: ' + row['bibliography_sources']
        else:
            return row['answer']
    
    # Definir categorías de características
    count_features = ['word_count', 'sentence_count']
    surface_features = [
        'avg_word_length', 'avg_sentence_length', 'reading_ease', 'grade_level',
        'connector_count'
    ]
    structure_features = [
        'unique_words_ratio', 'lexical_diversity',
        'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio', 'pronoun_ratio',
        'named_entity_count', 'dependency_distance'
    ]
    all_features = count_features + surface_features + structure_features
    
    # Formatear nombres de evaluadores para display
    formatted_eval_names = [format_label(name) for name in eval_names]
    
    # Para cada evaluador, calcular correlaciones
    all_correlations = {}
    all_lower_ci = {}
    all_upper_ci = {}
    
    for eval_collection, eval_name, formatted_name in zip(eval_collections, eval_names, formatted_eval_names):
        print(f"\nProcesando evaluaciones de {formatted_name}...")
        
        # Cargar evaluaciones
        evaluations = load_data(eval_collection)
        evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)
        
        # Cargar respuestas
        answers = load_data("answers")
        answers["_id"] = answers["_id"].astype(str)
        
        # Hacer merge con todas las columnas necesarias (igual que en tu código)
        evaluations_with_answers = evaluations.merge(
            answers[['_id', 'answer', 'bibliography_sources', 'confidence_level']],
            left_on='original_answer_id',
            right_on='_id',
            how='left'
        )
        
        # Crear la columna original_answer (igual que en tu código)
        evaluations_with_answers['original_answer'] = evaluations_with_answers.apply(
            combine_answer_and_bibliography, axis=1
        )
        
        # Filtrar IDK si es necesario
        if idk:
            try:
                evaluations_with_answers, _, _ = filter_and_analyze_responses(evaluations_with_answers)
            except Exception as e:
                print(f"Warning: Could not filter IDK responses for {formatted_name}: {e}")
        
        # Extraer características lingüísticas usando 'original_answer'
        print(f"Extrayendo características lingüísticas para {formatted_name}...")
        eval_with_features = prepare_dataframe_for_analysis(
            evaluations_with_answers, 
            text_column='original_answer'
        )
        
        # Calcular correlaciones para cada característica
        correlations_list = []
        lower_ci_list = []
        upper_ci_list = []
        
        for feature in all_features:
            if feature in eval_with_features.columns and score_column in eval_with_features.columns:
                result = bootstrap_correlation(
                    eval_with_features, feature, score_column, n_bootstrap, n_samples
                )
                correlations_list.append(result['correlation'])
                lower_ci_list.append(result['lower_ci'])
                upper_ci_list.append(result['upper_ci'])
            else:
                correlations_list.append(np.nan)
                lower_ci_list.append(np.nan)
                upper_ci_list.append(np.nan)
        
        # Usar el nombre formateado como clave
        all_correlations[formatted_name] = correlations_list
        all_lower_ci[formatted_name] = lower_ci_list
        all_upper_ci[formatted_name] = upper_ci_list
    
    # Crear DataFrames con resultados (con nombres formateados)
    point_correlations = pd.DataFrame(all_correlations, index=all_features)
    lower_ci_df = pd.DataFrame(all_lower_ci, index=all_features)
    upper_ci_df = pd.DataFrame(all_upper_ci, index=all_features)
    
    # Crear heatmap con clustering
    fig = create_clustered_linguistic_heatmap(
        point_correlations,
        lower_ci_df,
        upper_ci_df,
        count_features,
        surface_features,
        structure_features,
        confidence=confidence
    )
    
    return fig, point_correlations, lower_ci_df, upper_ci_df
def create_clustered_linguistic_heatmap(
    point_correlations,
    lower_ci,
    upper_ci,
    count_features,
    surface_features,
    structure_features,
    confidence=0.95
):
    """
    Creates a publication-quality correlation heatmap with confidence intervals and clustering.
    Nature style.
    """
    from scipy.cluster.hierarchy import linkage, dendrogram
    
    def format_label(label):
        """Format labels: remove underscores, capitalize properly"""
        # Replace underscores with spaces
        formatted = label.replace('_', ' ')
        # Capitalize each word
        formatted = ' '.join(word.capitalize() for word in formatted.split())
        return formatted
    
    # Cluster columns (evaluators)
    corr_matrix_for_clustering = point_correlations.T.fillna(0)
    
    if corr_matrix_for_clustering.shape[0] > 1:
        linkage_matrix = linkage(corr_matrix_for_clustering, method='ward')
        dendro = dendrogram(linkage_matrix, no_plot=True)
        cluster_order = dendro['leaves']
        sorted_columns = [corr_matrix_for_clustering.index[i] for i in cluster_order]
    else:
        sorted_columns = list(point_correlations.columns)
    
    sorted_corr = point_correlations[sorted_columns]
    sorted_lower = lower_ci[sorted_columns]
    sorted_upper = upper_ci[sorted_columns]
    
    # Determine significance
    significant = ((sorted_lower > 0) & (sorted_upper > 0)) | ((sorted_lower < 0) & (sorted_upper < 0))
    
    # Create annotations with correlations and significance markers
    annotations_text = []
    for i, feature in enumerate(sorted_corr.index):
        row_text = []
        for j, evaluator in enumerate(sorted_corr.columns):
            corr_val = sorted_corr.loc[feature, evaluator]
            is_sig = significant.loc[feature, evaluator]
            
            if np.isnan(corr_val):
                text = "—"
            else:
                text = f"{corr_val:.2f}"
                if is_sig:
                    text += "*"
            
            row_text.append(text)
        annotations_text.append(row_text)
    
    # Format labels for display
    formatted_y_labels = [format_label(feature) for feature in sorted_corr.index]
    formatted_x_labels = [format_label(evaluator) for evaluator in sorted_corr.columns]
    
    # Create base figure
    fig = go.Figure()
    
    # Add heatmap
    fig.add_trace(go.Heatmap(
        z=sorted_corr.values,
        x=formatted_x_labels,
        y=formatted_y_labels,
        colorscale='RdBu_r',
        zmid=0,
        zmin=-1,
        zmax=1,
        text=annotations_text,
        texttemplate='%{text}',
        textfont={"size": 18, "family": "Arial", "color": "black", "weight": "bold"},
        colorbar=dict(
            title=dict(
                text="<b>Correlation</b>",
                font=dict(size=22, family="Arial", color="black")
            ),
            tickfont=dict(size=18, family="Arial", color="black"),
            thickness=20,
            len=0.7,
            outlinecolor='black',
            outlinewidth=2
        ),
        hovertemplate='<b>Feature:</b> %{y}<br><b>Evaluator:</b> %{x}<br><b>Correlation:</b> %{z:.3f}<extra></extra>'
    ))
    
    # Add category separator lines
    count_end = len(count_features) - 0.5
    surface_end = count_end + len(surface_features)
    
    # Horizontal lines (thicker for Nature style)
    fig.add_shape(
        type="line",
        x0=-0.5, x1=len(sorted_corr.columns)-0.5,
        y0=count_end, y1=count_end,
        line=dict(color="black", width=3)
    )
    fig.add_shape(
        type="line",
        x0=-0.5, x1=len(sorted_corr.columns)-0.5,
        y0=surface_end, y1=surface_end,
        line=dict(color="black", width=3)
    )
    
    # Add category labels on the left
    category_positions = {
        'Count': len(count_features) / 2 - 0.5,
        'Surface': count_end + len(surface_features) / 2,
        'Structure': surface_end + len(structure_features) / 2
    }
    
    # Update layout for Nature style
    fig.update_layout(
        title=dict(
            text="<b>Linguistic Features vs Evaluator Accuracy Scores</b>",
            font=dict(size=32, family="Arial", color="black"),
            x=0.5,
            xanchor='center',
            y=0.98,
            yanchor='top'
        ),
        xaxis=dict(
            title=dict(
                text="<b>Evaluators (Clustered)</b>",
                font=dict(size=26, family="Arial", color="black"),
                standoff=10
            ),
            tickfont=dict(size=22, family="Arial", color="black"),
            side='bottom',
            showgrid=False,
            zeroline=False,
            linecolor='black',
            linewidth=2,
            tickangle=-45
        ),
        yaxis=dict(
            title=dict(
                text="",
                font=dict(size=26, family="Arial", color="black")
            ),
            tickfont=dict(size=22, family="Arial", color="black"),
            autorange='reversed',
            showgrid=False,
            zeroline=False,
            linecolor='black',
            linewidth=2
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=800,
        height=800,
        margin=dict(l=280, r=180, t=120, b=150)
    )
    
    return fig



In [ ]:
fig_heatmap, point_corr, lower_ci, upper_ci = create_all_evaluators_linguistic_heatmap_with_bootstrap(
    eval_collections=EVAL_COLLECTION_LIST,
    eval_names=EVAL_NAMES,
    score_column='accuracy_score',
    n_bootstrap=1000,
    n_samples=50,
    confidence=0.95,
    idk=True
)
fig_heatmap.show()


In [ ]:
fig_heatmap.write_image("linguistic_heatmap.svg", scale=3)

In [ ]:
def create_linguistic_features_violin_plots(
    eval_collection='ia_eval',
    features_to_plot=None,
    text_column='answer',
    height=800,
    width=2400,
    ncols=4,
    pvalue_correction='fdr_bh'  # 'fdr_bh', 'bonferroni', 'holm', etc.
):
    """
    Crea violin plots comparando características lingüísticas entre los tres tipos 
    de respondedores (Human=0, AI=1, AI CoT=2).
    Usa Mann-Whitney por pares con corrección FDR.
    Filtra respuestas "I don't know".
    """
    from plotly.subplots import make_subplots
    from scipy.stats import mannwhitneyu
    from statsmodels.stats.multitest import multipletests
    import numpy as np
    
    print(f"Cargando evaluaciones de {eval_collection}...")
    
    # Cargar evaluaciones (tienen LLM)
    evaluations = load_data(eval_collection)
    evaluations["original_answer_id"] = evaluations["original_answer_id"].astype(str)
    
    # Cargar respuestas
    answers = load_data("answers")
    answers["_id"] = answers["_id"].astype(str)
    
    # Merge: evaluaciones + respuestas
    df = evaluations.merge(
        answers[['_id', text_column]],
        left_on='original_answer_id',
        right_on='_id',
        how='left'
    )
    
    # Verificar que LLM existe
    if 'LLM' not in df.columns:
        print("Error: La columna 'LLM' no existe en las evaluaciones")
        print(f"Columnas disponibles: {list(df.columns)}")
        return None
    
    # Filtrar solo las que tienen LLM definido
    df = df[df['LLM'].notna()].copy()
    
    print(f"\nRespuestas por tipo de LLM antes de filtrar:")
    print(df['LLM'].value_counts().sort_index())
    
    # FILTRAR "I DON'T KNOW"
    try:
        df['original_answer'] = df[text_column]
    except:
        pass
    
    print("\nFiltrando respuestas 'I don't know'...")
    df_filtered, summary, detailed_counts = filter_and_analyze_responses(df)
    
    print(f"\nRespuestas después de filtrar 'I don't know':")
    print(f"  Total antes: {len(df)}")
    print(f"  Total después: {len(df_filtered)}")
    print(f"  Eliminadas: {len(df) - len(df_filtered)}")
    
    print("\nRespuestas por tipo de LLM después de filtrar:")
    print(df_filtered['LLM'].value_counts().sort_index())
    
    # Extraer características lingüísticas
    print("\nExtrayendo características lingüísticas...")
    df_with_features = prepare_dataframe_for_analysis(df_filtered, text_column=text_column)
    
    # Características por defecto
    if features_to_plot is None:
        features_to_plot = [
            'word_count', 'sentence_count', 'avg_word_length', 'avg_sentence_length',
            'reading_ease', 'grade_level', 'unique_words_ratio', 'lexical_diversity',
            'verb_ratio', 'noun_ratio', 'adj_ratio', 'adv_ratio',
            'named_entity_count', 'dependency_distance', 'connector_count', 'pronoun_ratio'
        ]
    
    # Calcular número de filas necesarias
    nrows = int(np.ceil(len(features_to_plot) / ncols))
    
    # Crear subplots
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=[f"<b>{feat.replace('_', ' ').title()}</b>" for feat in features_to_plot],
        vertical_spacing=0.12,
        horizontal_spacing=0.05
    )
    
    # Etiquetas y colores
    responder_labels = {0: 'Human', 1: 'AI', 2: 'AI CoT'}
    responder_colors = {
        'Human': 'rgb(31, 119, 180)',
        'AI': 'rgb(255, 127, 14)',
        'AI CoT': 'rgb(44, 160, 44)'
    }
    
    # Recoger todos los p-values para ajuste global
    all_pvalues = []
    pvalue_info = []  # Guardar info de cada test
    
    # Para cada característica
    for idx, feature in enumerate(features_to_plot):
        row = idx // ncols + 1
        col = idx % ncols + 1
        
        if feature not in df_with_features.columns:
            print(f"Warning: Feature '{feature}' not found")
            continue
        
        # Preparar datos por tipo
        feature_data = {}
        for llm_type in [0, 1, 2]:
            data = df_with_features[df_with_features['LLM'] == llm_type][feature].dropna()
            label = responder_labels[llm_type]
            feature_data[label] = data.values
            
            # Añadir violin
            fig.add_trace(
                go.Violin(
                    y=data,
                    name=label,
                    legendgroup=label,
                    scalegroup=label,
                    showlegend=(idx == 0),
                    line_color=responder_colors[label],
                    fillcolor=responder_colors[label],
                    opacity=0.6,
                    meanline_visible=True,
                    box_visible=True,
                    points=False
                ),
                row=row,
                col=col
            )
        
        # Mann-Whitney por pares (sin ajustar aún)
        try:
            if all(len(feature_data[k]) > 0 for k in feature_data.keys()):
                # Human vs AI
                stat1, pval1 = mannwhitneyu(feature_data['Human'], feature_data['AI'], alternative='two-sided')
                all_pvalues.append(pval1)
                pvalue_info.append({'feature_idx': idx, 'comparison': 'Human-AI', 'pvalue_raw': pval1})
                
                # Human vs CoT
                stat2, pval2 = mannwhitneyu(feature_data['Human'], feature_data['AI CoT'], alternative='two-sided')
                all_pvalues.append(pval2)
                pvalue_info.append({'feature_idx': idx, 'comparison': 'Human-CoT', 'pvalue_raw': pval2})
                
                # AI vs CoT
                stat3, pval3 = mannwhitneyu(feature_data['AI'], feature_data['AI CoT'], alternative='two-sided')
                all_pvalues.append(pval3)
                pvalue_info.append({'feature_idx': idx, 'comparison': 'AI-CoT', 'pvalue_raw': pval3})
                
        except Exception as e:
            print(f"Could not calculate Mann-Whitney for {feature}: {e}")
        
        # Añadir n
        for llm_idx, llm_type in enumerate([0, 1, 2]):
            label = responder_labels[llm_type]
            n = len(feature_data[label])
            
            xref = f"x{(row-1)*ncols + col}" if ((row-1)*ncols + col) > 1 else "x"
            yref = f"y{(row-1)*ncols + col}" if ((row-1)*ncols + col) > 1 else "y"
            
            y_min = min([feature_data[k].min() for k in feature_data.keys() if len(feature_data[k]) > 0])
            
            fig.add_annotation(
                text=f"n={n}",
                xref=xref,
                yref=yref,
                x=llm_idx,
                y=y_min * 1.1 if y_min < 0 else y_min * 0.9,
                showarrow=False,
                font=dict(size=11, family="Arial", color="gray")
            )
    
    # AJUSTE DE P-VALUES (FDR)
    print(f"\nAjustando {len(all_pvalues)} p-values con {pvalue_correction}...")
    if len(all_pvalues) > 0:
        reject, pvals_corrected, _, _ = multipletests(all_pvalues, alpha=0.05, method=pvalue_correction)
        
        # Actualizar pvalue_info con p-values ajustados
        for i, info in enumerate(pvalue_info):
            info['pvalue_adjusted'] = pvals_corrected[i]
            info['significant'] = reject[i]
        
        print(f"P-values significativos después de corrección: {sum(reject)}/{len(all_pvalues)}")
    
    # Función para formatear p-value
    def format_pval(p):
        if np.isnan(p):
            return "ns"
        elif p < 0.001:
            return "p<0.001"
        else:
            return f"p={p:.3f}"
    
    # AÑADIR BRACKETS CON P-VALUES AJUSTADOS
    for idx, feature in enumerate(features_to_plot):
        if feature not in df_with_features.columns:
            continue
            
        row = idx // ncols + 1
        col = idx % ncols + 1
        
        # Obtener p-values para esta característica
        feature_pvals = [p for p in pvalue_info if p['feature_idx'] == idx]
        
        if len(feature_pvals) == 0:
            continue
        
        # Preparar datos para calcular y_max
        feature_data_for_scale = {}
        for llm_type in [0, 1, 2]:
            data = df_with_features[df_with_features['LLM'] == llm_type][feature].dropna()
            label = responder_labels[llm_type]
            feature_data_for_scale[label] = data.values
        
        y_max = max([feature_data_for_scale[k].max() for k in feature_data_for_scale.keys()])
        y_min = min([feature_data_for_scale[k].min() for k in feature_data_for_scale.keys()])
        
        xref = f"x{(row-1)*ncols + col}" if ((row-1)*ncols + col) > 1 else "x"
        yref = f"y{(row-1)*ncols + col}" if ((row-1)*ncols + col) > 1 else "y"
        
        # Encontrar p-values ajustados para cada comparación
        pval_human_ai = next((p['pvalue_adjusted'] for p in feature_pvals if p['comparison'] == 'Human-AI'), np.nan)
        pval_human_cot = next((p['pvalue_adjusted'] for p in feature_pvals if p['comparison'] == 'Human-CoT'), np.nan)
        pval_ai_cot = next((p['pvalue_adjusted'] for p in feature_pvals if p['comparison'] == 'AI-CoT'), np.nan)
        
        # Human vs AI (bracket inferior)
        y_h1 = y_max * 1.05
        fig.add_shape(
            type="line",
            xref=xref, yref=yref,
            x0=0, x1=1,
            y0=y_h1, y1=y_h1,
            line=dict(color="black", width=1.5)
        )
        fig.add_shape(type="line", xref=xref, yref=yref, x0=0, x1=0, y0=y_h1, y1=y_h1-0.02*(y_max-y_min), line=dict(color="black", width=1.5))
        fig.add_shape(type="line", xref=xref, yref=yref, x0=1, x1=1, y0=y_h1, y1=y_h1-0.02*(y_max-y_min), line=dict(color="black", width=1.5))
        fig.add_annotation(
            text=format_pval(pval_human_ai),
            xref=xref, yref=yref,
            x=0.5, y=y_h1 + 0.1*(y_max-y_min),
            showarrow=False,
            font=dict(size=11, family="Arial", color="black")
        )
        
        # AI vs CoT (bracket medio)
        y_h2 = y_max * 1.14
        fig.add_shape(
            type="line",
            xref=xref, yref=yref,
            x0=1, x1=2,
            y0=y_h2, y1=y_h2,
            line=dict(color="black", width=1.5)
        )
        fig.add_shape(type="line", xref=xref, yref=yref, x0=1, x1=1, y0=y_h2, y1=y_h2-0.02*(y_max-y_min), line=dict(color="black", width=1.5))
        fig.add_shape(type="line", xref=xref, yref=yref, x0=2, x1=2, y0=y_h2, y1=y_h2-0.02*(y_max-y_min), line=dict(color="black", width=1.5))
        fig.add_annotation(
            text=format_pval(pval_ai_cot),
            xref=xref, yref=yref,
            x=1.5, y=y_h2 + 0.1*(y_max-y_min),
            showarrow=False,
            font=dict(size=11, family="Arial", color="black")
        )
        
        # Human vs CoT (bracket superior)
        y_h3 = y_max * 1.38
        fig.add_shape(
            type="line",
            xref=xref, yref=yref,
            x0=0, x1=2,
            y0=y_h3, y1=y_h3,
            line=dict(color="black", width=1.5)
        )
        fig.add_shape(type="line", xref=xref, yref=yref, x0=0, x1=0, y0=y_h3, y1=y_h3-0.02*(y_max-y_min), line=dict(color="black", width=1.5))
        fig.add_shape(type="line", xref=xref, yref=yref, x0=2, x1=2, y0=y_h3, y1=y_h3-0.02*(y_max-y_min), line=dict(color="black", width=1.5))
        fig.add_annotation(
            text=format_pval(pval_human_cot),
            xref=xref, yref=yref,
            x=1, y=y_h3 + 0.1*(y_max-y_min),
            showarrow=False,
            font=dict(size=11, family="Arial", color="black")
        )
    
    # Layout
    fig.update_layout(
        title=dict(
            text="<b>Linguistic Features Comparison Across Responder Types</b>",
            font=dict(size=32, family="Arial", color="black"),
            x=0.5,
            xanchor='center'
        ),
        showlegend=True,
        legend=dict(
            font=dict(size=18, family="Arial"),
            x=1.01,
            y=1,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='black',
            borderwidth=1
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        height=height,
        width=width,
        margin=dict(l=60, r=150, t=100, b=60)
    )
    
    # Update ejes
    for i in range(1, nrows * ncols + 1):
        row = (i - 1) // ncols + 1
        col = (i - 1) % ncols + 1
        
        fig.update_xaxes(
            showgrid=False,
            zeroline=False,
            linecolor='black',
            linewidth=2,
            tickfont=dict(size=14, family="Arial"),
            row=row, col=col
        )
        
        fig.update_yaxes(
            showgrid=False,
            zeroline=False,
            linecolor='black',
            linewidth=2,
            tickfont=dict(size=14, family="Arial"),
            row=row, col=col
        )
    
    # Update títulos
    for annotation in fig['layout']['annotations']:
        if 'text' in annotation and annotation['text'].startswith('<b>'):
            annotation['font'] = dict(size=16, family="Arial", color="black")
    
    return fig


In [ ]:
# Crear violin plots de características lingüísticas
# Comparando Human vs AI vs AI CoT
fig_linguistic = create_linguistic_features_violin_plots(
    features_to_plot=None,  # Usa todas las características por defecto
    text_column='answer',   # Columna con el texto de las respuestas
    height=800,
    width=1200,
    ncols=4  # 4 columnas de subplots
)
fig_linguistic.show()